# 🌐 Multilingual Chat Translator v2
### Upgraded Pipeline: Hybrid Language Detection + Smart Routing + Context-Aware Normalization + BERTScore Evaluation

**Pipeline Overview:**
```
raw input
    ↓
[1] Hybrid Language Detection  (Script regex → IndicLID → fallback)
    ↓
[2] Smart Routing              (Heuristic informality scorer)
    → FORMAL + NATIVE SCRIPT: skip normalization
    → INFORMAL / ROMANIZED / CODE-MIX: normalize with context
    ↓
[3] Context-Aware Normalization (Mistral + conversation history + speaker identity)
    ↓
[4] IndicTrans2 Translation
    ↓
[5] BERTScore Evaluation        (evaluation mode only)
    ↓
Rich output dict
```


In [ ]:
!pip install datasets -q

In [ ]:
!git clone https://github.com/microsoft/GLUECoS.git
!ls GLUECoS/

Cloning into 'GLUECoS'...
remote: Enumerating objects: 455, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 455 (delta 111), reused 108 (delta 108), pack-reused 335 (from 1)
Receiving objects: 100% (455/455), 815.91 KiB | 5.20 MiB/s, done.
Resolving deltas: 100% (231/231), done.
all_roman.txt	     CODE_OF_CONDUCT.md  LICENSE	   SECURITY.md
azure_ml	     Data		 NOTICE		   train.sh
azure-pipelines.yml  docs		 README.md	   transliterator.py
Code		     download_data.sh	 requirements.txt


In [ ]:
!ls GLUECoS/Data/

Original_Data  Preprocess_Scripts  README.md


In [ ]:
!ls GLUECoS/Data/Original_Data/

LID_EN_ES  NER_EN_HI  POS_EN_ES     QA_EN_HI	     Sentiment_EN_HI
NER_EN_ES  NLI_EN_HI  POS_EN_HI_UD  Sentiment_EN_ES


In [ ]:
from datasets import load_dataset

FLORES_PAIRS = [
    ("tam_Taml", "hin_Deva", "tamil",     "hindi"),
    ("kan_Knda", "hin_Deva", "kannada",   "hindi"),
    ("tel_Telu", "tam_Taml", "telugu",    "tamil"),
    ("mal_Mlym", "kan_Knda", "malayalam", "kannada"),
    ("ben_Beng", "hin_Deva", "bengali",   "hindi"),
]

flores_sentences = []

for src_code, tgt_code, src_lang, tgt_lang in FLORES_PAIRS:
    src_data = load_dataset("openlanguagedata/flores_plus", src_code, split="devtest")
    tgt_data = load_dataset("openlanguagedata/flores_plus", tgt_code, split="devtest")

    for src_row, tgt_row in list(zip(src_data, tgt_data))[:6]:
        flores_sentences.append({
            "input":     src_row["text"],
            "reference": tgt_row["text"],
            "src_lang":  src_lang,
            "tgt_lang":  tgt_lang,
        })

    print(f"✅ Loaded {src_lang} → {tgt_lang}")

print(f"\nTotal FLORES sentences collected: {len(flores_sentences)}")

# Preview
for s in flores_sentences[:3]:
    print(f"\n[{s['src_lang']} → {s['tgt_lang']}]")
    print(f"  Input:     {s['input']}")
    print(f"  Reference: {s['reference']}")

Resolving data files:   0%|          | 0/224 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/219 [00:00<?, ?it/s]

tam_Taml.jsonl:   0%|          | 0.00/696k [00:00<?, ?B/s]

tam_Taml.jsonl:   0%|          | 0.00/730k [00:00<?, ?B/s]

Generating dev split: 0 examples [00:00, ? examples/s]

...

Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

Resolving data files:   0%|          | 0/224 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/219 [00:00<?, ?it/s]

✅ Loaded bengali → hindi

Total FLORES sentences collected: 30

[tamil → hindi]
  Input:     """எங்களிடம் இப்போது  4-மாத-வயதுடைய எலி ஒன்று உள்ளது, முன்னர் அதற்கு நீரிழிவு இருந்தது தற்போது இல்லை"" என்று அவர் மேலும் கூறினார்."
  Reference: उन्होंने कहा “कि अब हमारे पास 4 महीने उम्र वाले चूहे हैं जिन्हें मधुमेह नहीं है जो मधुमेह के रोगी थे। ”

[tamil → hindi]
  Input:     ஹாலிஃபாக்ஸில் உள்ள டல்ஹெளசி பல்கலைக்கழகத்தின் மருத்துவப் பேராசிரியரும், நோவா ஸ்கோட்டியா மற்றும் கனேடிய நீரிழிவு சங்கத்தின் மருத்துவ மற்றும் அறிவியல் பிரிவின் தலைவருமான டாக்டர் இஹுத் உர், ஆராய்ச்சி இன்னும் ஆரம்ப நாட்களில்தான் உள்ளது என்று எச்சரித்துள்ளார்.
  Reference: डॉ. एहुड उर, नोवा स्कोटिया के हैलिफ़ैक्स में डलहौज़ी विश्वविद्यालय में चिकित्सा के प्रोफ़ेसर और कनाडाई डायबिटीज़ एसोसिएशन के नैदानिक ​​और वैज्ञानिक विभाग के अध्यक्ष ने आगाह किया कि शोध अभी भी अपने शुरुआती दिनों में है.

[tamil → hindi]
  Input:     பிற வல்லுனர்கள் போலவே அவரும், ஏற்கனவே டைப் 1 இரத்த சர்க்கரை நோயுள்ளவர்களுக்கு இந்த கண்டுபிடிப்புகள் பொருந்தவில

---
## 0. Authentication & Base Installs

In [ ]:
!pip install -q \
  "transformers==4.43.0" \
  "torch==2.3.0" \
  "IndicTransToolkit" \
  "sentencepiece" \
  "sacremoses"

# Then patch collator
collator_path = "/usr/local/lib/python3.12/dist-packages/IndicTransToolkit/collator.py"
with open(collator_path, "r") as f:
    content = f.read()
content = content.replace(
    "from transformers.tokenization_utils import PreTrainedTokenizerBase",
    "from transformers.tokenization_utils_base import PreTrainedTokenizerBase"
)
with open(collator_path, "w") as f:
    f.write(content)

print("✅ Done — now restart runtime")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 761.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 822.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from huggingface_hub import login
login()

In [ ]:
# Core dependencies
!pip -q install -U \
  "opentelemetry-api==1.38.0" \
  "opentelemetry-sdk==1.38.0" \
  "opentelemetry-semantic-conventions==0.59b0" \
  "opentelemetry-exporter-otlp-proto-http==1.38.0"

# New in v2
!pip -q install -U \
  "bert-score" \
  "fasttext-wheel" \
  "mistralai==0.4.2"

!pip install -q groq

# Clone IndicLID inference code (weights loaded separately from Drive)
import os
if not os.path.exists('IndicLID'):
    !git clone https://github.com/AI4Bharat/IndicLID.git
    print('✅ IndicLID repo cloned')
else:
    print('✅ IndicLID repo already present')

print('✅ Installs complete')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 884.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.1/133.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 6.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-sdk 0.3.9 requires orjson>=3.11.5, but you have orjson 3.10.18 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 6.1 MB/s eta 0:00:00
Cloning into 'IndicLID'...
remote: Enumerating objects: 345, done.
remote: Counting objects: 100% (345/345), done.
remote: Compressing objects: 100% (191/191), done.
remote: Total 345 (delta 151), reused 291 (delta 118), pack-reused 0 (from 0)
Receiving

In [ ]:
import torch
import transformers
import mistralai

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("mistralai:", getattr(mistralai, "__version__", "ok"))

from mistralai.client import MistralClient
print("✅ MistralClient import OK")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

torch: 2.3.0+cu121
transformers: 4.43.0
mistralai: ok
✅ MistralClient import OK


---
## 1. Load IndicTrans2 (Translation Model — unchanged from v1)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from IndicTransToolkit import IndicProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

INDICTRANS_MODEL_ID = "ai4bharat/indictrans2-en-indic-1B"

indic_tokenizer = AutoTokenizer.from_pretrained(
    INDICTRANS_MODEL_ID,
    trust_remote_code=True
)

indic_model = AutoModelForSeq2SeqLM.from_pretrained(
    INDICTRANS_MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None
)

if DEVICE != "cuda":
    indic_model = indic_model.to(DEVICE)

indic_model.eval()
ip = IndicProcessor(inference=True)

print("✅ IndicTrans2 loaded")
INDICTRANS_REVERSE_MODEL_ID = "ai4bharat/indictrans2-indic-en-1B"

reverse_tokenizer = AutoTokenizer.from_pretrained(
    INDICTRANS_REVERSE_MODEL_ID,
    trust_remote_code=True
)

reverse_model = AutoModelForSeq2SeqLM.from_pretrained(
    INDICTRANS_REVERSE_MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None
)

if DEVICE != "cuda":
    reverse_model = reverse_model.to(DEVICE)

reverse_model.eval()
print("✅ IndicTrans2 reverse model loaded")
print("✅ Translation function ready")


DEVICE: cpu


tokenizer_config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

tokenization_indictrans.py:   0%|          | 0.00/8.04k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-1B:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json:   0%|          | 0.00/645k [00:00<?, ?B/s]

...

modeling_indictrans.py:   0%|          | 0.00/79.8k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-1B:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/4.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

✅ IndicTrans2 reverse model loaded
✅ Translation function ready


In [ ]:
# Language code map
LANG_CODE = {
    "hindi":     "hin_Deva",
    "kannada":   "kan_Knda",
    "tamil":     "tam_Taml",
    "telugu":    "tel_Telu",
    "malayalam": "mal_Mlym",
    "marathi":   "mar_Deva",
    "bengali":   "ben_Beng",
    "gujarati":  "guj_Gujr",
    "punjabi":   "pan_Guru",
    "odia":      "ory_Orya",
    "english":   "eng_Latn",
}

# Indic → English source codes (for pivot translation)
INDIC_TO_ENG_CODE = {
    "hindi":     "hin_Deva",
    "kannada":   "kan_Knda",
    "tamil":     "tam_Taml",
    "telugu":    "tel_Telu",
    "malayalam": "mal_Mlym",
    "marathi":   "mar_Deva",
    "bengali":   "ben_Beng",
    "gujarati":  "guj_Gujr",
    "punjabi":   "pan_Guru",
    "odia":      "ory_Orya",
}

def translate_with_indictrans(
    sentences,
    src_lang="eng_Latn",
    tgt_lang="hin_Deva",
    max_length=256,
    num_beams=5
):
    batch = ip.preprocess_batch(sentences, src_lang=src_lang, tgt_lang=tgt_lang, visualize=False)

    inputs = indic_tokenizer(
        batch,
        truncation=True,
        padding="longest",
        max_length=max_length,
        return_tensors="pt",
        return_attention_mask=True
    ).to(DEVICE)

    with torch.inference_mode():
        generated = indic_model.generate(
            **inputs,
            use_cache=True,
            min_length=0,
            max_length=max_length,
            num_beams=num_beams,
            num_return_sequences=1
        )

    decoded = indic_tokenizer.batch_decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )
    return ip.postprocess_batch(decoded, lang=tgt_lang)

def translate_indic_to_english(sentences, src_lang="hin_Deva"):
    batch = ip.preprocess_batch(
        sentences,
        src_lang=src_lang,
        tgt_lang="eng_Latn",
        visualize=False
    )
    inputs = reverse_tokenizer(
        batch,
        truncation=True,
        padding="longest",
        max_length=256,
        return_tensors="pt",
        return_attention_mask=True
    ).to(DEVICE)
    with torch.inference_mode():
        generated = reverse_model.generate(
            **inputs,
            use_cache=True,
            min_length=0,
            max_length=256,
            num_beams=5,
            num_return_sequences=1
        )
    decoded = reverse_tokenizer.batch_decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )
    return ip.postprocess_batch(decoded, lang="eng_Latn")



In [ ]:
def translate_indic_to_indic(
    text: str,
    src_language: str,
    tgt_language: str
) -> tuple[str, str]:
    """
    Translates between two Indic languages using English as pivot.

    Indic → English → Indic

    Returns:
        (final_translation, english_pivot)
    """
    src_code = INDIC_TO_ENG_CODE.get(src_language)
    tgt_code = LANG_CODE.get(tgt_language)

    if not src_code:
        raise ValueError(f"Unsupported source language: {src_language}")
    if not tgt_code:
        raise ValueError(f"Unsupported target language: {tgt_language}")

    # Step 1: Indic → English
    english_pivot = translate_indic_to_english([text], src_lang=src_code)[0]


    # Step 2: English → Target Indic
    final = translate_with_indictrans(
        [english_pivot],
        src_lang="eng_Latn",
        tgt_lang=tgt_code
    )[0]

    return final, english_pivot



# Quick test
result, pivot = translate_indic_to_indic("சர்வர் டவுன் ஆகிவிட்டது", "tamil", "hindi")
print(f"Tamil → English pivot: {pivot}")
print(f"Tamil → Hindi final:   {result}")

Tamil → English pivot: The server is down
Tamil → Hindi final:   सर्वर डाउन है


In [ ]:
# Test each language pair individually
test_pairs = [
    ("சர்வர் டவுன் ஆகிவிட்டது", "tam_Taml", "Tamil"),
    ("ಸರ್ವರ್ ಡೌನ್ ಆಗಿದೆ", "kan_Knda", "Kannada"),
    ("সার্ভার ডাউন হয়েছে", "ben_Beng", "Bengali"),
    ("सर्वर डाउन हो गया है", "hin_Deva", "Hindi"),
    ("సర్వర్ డౌన్ అయింది", "tel_Telu", "Telugu"),
]

for text, code, lang in test_pairs:
    result = translate_indic_to_english([text], src_lang=code)[0]
    print(f"{lang}: {result}")

Tamil: The server is down
Kannada: The server is down
Bengali: The server is down.
Hindi: The server is down
Telugu: The server is down


---
## 2. Module 1 — Hybrid Language Detector

**Three-stage strategy:**
1. **Script detection** via Unicode range regex — deterministic, instant, 100% reliable for native-script text
2. **IndicLID** (AI4Bharat) — handles all 22 Indian languages in both native and romanized script, including Kanglish, Tanglish, Hinglish
3. **Fallback** — treat as English if IndicLID confidence is low

**Why keep script detection at Stage 1 if IndicLID handles native script too?**
Script regex is instant and deterministic. For unambiguous scripts (Tamil, Telugu, Kannada etc.) there is zero reason to call a model. IndicLID is only invoked when script detection is inconclusive — i.e. for Devanagari (Hindi/Marathi ambiguity) and all romanized/code-mixed text. This is a deliberate efficiency decision.

> **⚠️ Before running this cell:** Make sure you have completed the model download steps
> described in the setup instructions and mounted your Google Drive.


In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
import re
import sys
import os
import unicodedata

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

MODELS_BASE = '/content/drive/MyDrive/IndicLID_models'
FTN_PATH    = os.path.join(MODELS_BASE, 'indiclid-ftn')
FTR_PATH    = os.path.join(MODELS_BASE, 'indiclid-ftr')
BERT_PATH   = os.path.join(MODELS_BASE, 'indiclid-bert')

for name, path in [('FTN', FTN_PATH), ('FTR', FTR_PATH), ('BERT', BERT_PATH)]:
    if os.path.exists(path):
        print(f'  ✅ {name} found at {path}')
    else:
        print(f'  ❌ {name} NOT FOUND at {path} — check your Drive folder structure')

models_dir = 'IndicLID/Inference/models'
os.makedirs(models_dir, exist_ok=True)
for folder in ['indiclid-ftn', 'indiclid-ftr', 'indiclid-bert']:
    src  = f'{MODELS_BASE}/{folder}'
    dest = f'{models_dir}/{folder}'
    if not os.path.exists(dest):
        os.symlink(src, dest)
        print(f'  ✅ Linked {folder}')
    else:
        print(f'  ✅ {folder} already linked')

sys.path.insert(0, 'IndicLID/Inference')
from ai4bharat.IndicLID import IndicLID

os.chdir('IndicLID/Inference')
indic_lid = IndicLID(input_threshold=0.5, roman_lid_threshold=0.6)
os.chdir('/content')
print('\n✅ IndicLID loaded successfully')


INDICLID_TO_NAME = {
    'hin': 'hindi',     'kan': 'kannada',   'tam': 'tamil',
    'tel': 'telugu',    'mal': 'malayalam', 'mar': 'marathi',
    'ben': 'bengali',   'guj': 'gujarati',  'pan': 'punjabi',
    'ori': 'odia',      'eng': 'english',   'asm': 'assamese',
    'urd': 'urdu',      'kas': 'kashmiri',  'kok': 'konkani',
    'mai': 'maithili',  'mni': 'manipuri',  'nep': 'nepali',
    'san': 'sanskrit',  'sat': 'santali',   'snd': 'sindhi',
    'brx': 'bodo',      'doi': 'dogri',
}

def _build_script_pattern(lang_name: str) -> str | None:
    SCRIPT_BLOCK_KEYWORDS = {
        'hindi':     'DEVANAGARI',
        'marathi':   'DEVANAGARI',
        'bengali':   'BENGALI',
        'gujarati':  'GUJARATI',
        'punjabi':   'GURMUKHI',
        'odia':      'ORIYA',
        'tamil':     'TAMIL',
        'telugu':    'TELUGU',
        'kannada':   'KANNADA',
        'malayalam': 'MALAYALAM',
    }
    keyword = SCRIPT_BLOCK_KEYWORDS.get(lang_name)
    if not keyword:
        return None

    in_range = False
    ranges = []
    start = None

    for cp in range(0x0000, 0x10000):
        char_name = unicodedata.name(chr(cp), '')
        belongs = keyword in char_name
        if belongs and not in_range:
            start = cp
            in_range = True
        elif not belongs and in_range:
            ranges.append((start, cp - 1))
            in_range = False
    if in_range:
        ranges.append((start, 0xFFFF))

    if not ranges:
        return None

    parts = [f'\\u{lo:04X}-\\u{hi:04X}' for lo, hi in ranges]
    return f'[{"".join(parts)}]'


def _build_all_script_patterns() -> dict:
    languages = [
        'hindi', 'marathi', 'bengali', 'gujarati', 'punjabi',
        'odia', 'tamil', 'telugu', 'kannada', 'malayalam'
    ]
    return {lang: pat for lang in languages if (pat := _build_script_pattern(lang))}


# Build dynamically at startup — no hardcoded hex ranges
SCRIPT_RANGES = _build_all_script_patterns()
print(f"✅ Script patterns built dynamically for: {list(SCRIPT_RANGES.keys())}")

UNAMBIGUOUS_SCRIPTS = {
    'bengali', 'gujarati', 'punjabi', 'odia',
    'tamil', 'telugu', 'kannada', 'malayalam'
}

# Word hints for romanized detection
# Only languages that commonly appear in romanized chat form
# Bengali, Gujarati, Punjabi (Gurmukhi), Odia etc. are rare in romanized chat
# and IndicLID handles them well enough
ROMANIZED_HINTS = {
    'hindi':     {'yaar', 'kya', 'hai', 'aur', 'aaj', 'karo', 'raha',
                  'rahi', 'gaya', 'hoga', 'bhi', 'toh', 'abhi', 'mera', 'tera',
                  'kuch', 'thoda', 'jaldi', 'bhej', 'ho', 'haan', 'kyun',
                  'matlab', 'samjha', 'dekho', 'suno', 'bolo', 'chalega'},
    'marathi':   {'aho', 'nako', 'ahe', 'mhanje', 'karay', 'bagh', 'sangto',
                  'yeto', 'jato', 'aahe', 'naahi', 'kasa', 'bara', 'kay',
                  'tumi', 'amhi', 'tumhi', 'chala', 'paha', 'sanga'},
    'kannada':   {'swalpa', 'maadi', 'illa', 'hogbeku', 'ide', 'aitu', 'ivathu',
                  'naale', 'madana', 'madthara','baralla', 'aagide', 'yen',
                  'madbeku', 'aagutte', 'madidini', 'barthini', 'banni', 'heli'},
    'tamil':     {'pannunga', 'irukku', 'poganuma', 'varala', 'panniten', 'innaiku',
                  'aaguthu', 'aaguren', 'aagala', 'pannuvana', 'romba', 'konjam',
                  'kal', 'panna', 'sollu', 'paaru', 'vanga', 'pongo', 'theriyum'},
    'telugu':    {'cheyyi', 'undi', 'ledu', 'avutundi', 'cheppandi', 'enduku',
                  'chudandi', 'vasthanu', 'ikkade', 'akkade', 'kaadu', 'ayindi',
                  'cheyyadam', 'vellandi', 'randi', 'chusthe'},
    'malayalam': {'alle', 'aano', 'undo', 'ille', 'cheyyam', 'varu', 'povo',
                  'nokku', 'paranju', 'ayyo', 'enthina', 'evide', 'ingane',
                  'cheyyu', 'parayoo', 'kandoo', 'venda', 'mathi'},
    'punjabi':   {'oye', 'sanu', 'tenu', 'assi', 'tussi', 'kehra', 'kiddan',
                  'kiven', 'changa', 'theek', 'dass', 'kar', 'nahin', 'haan',
                  'panga', 'yaar', 'paaji', 'veere', 'kudi', 'munda'},
}


def detect_by_script(text: str) -> str | None:
    counts = {}
    for lang, pattern in SCRIPT_RANGES.items():
        n = len(re.findall(pattern, text))
        if n > 0:
            counts[lang] = n
    if not counts:
        return None
    dominant = max(counts, key=counts.get)
    if dominant in ('hindi', 'marathi'):
        return None
    return dominant if dominant in UNAMBIGUOUS_SCRIPTS else None


def detect_romanized_language(text: str) -> str | None:
    # Strip punctuation from tokens for cleaner matching
    tokens = set(re.sub(r'[^\w\s]', '', text.lower()).split())
    scores = {}
    for lang, wordlist in ROMANIZED_HINTS.items():
        hits = tokens & wordlist
        if hits:
            scores[lang] = len(hits)
    if not scores:
        return None
    return max(scores, key=scores.get)


def detect_by_indiclid(text: str) -> tuple[str, float, str]:
    # Word-list pre-check — more reliable than IndicLID for short romanized text
    hint_lang = detect_romanized_language(text)
    if hint_lang:
        code_map = {
            'hindi':     'hin_Latn', 'kannada':   'kan_Latn',
            'tamil':     'tam_Latn', 'telugu':    'tel_Latn',
            'marathi':   'mar_Latn', 'malayalam': 'mal_Latn',
            'punjabi':   'pan_Latn',
        }
        return hint_lang, 1.0, code_map.get(hint_lang, 'hin_Latn')

    # Fall through to IndicLID for everything else
    result = indic_lid.batch_predict([text], batch_size=1)
    if not result:
        return 'english', 0.0, 'eng_Latn'

    _, lang_code, score, model_used = result[0]
    lang_prefix = lang_code.split('_')[0].lower()
    lang_name = INDICLID_TO_NAME.get(lang_prefix, 'english')
    return lang_name, float(score), lang_code


def detect_language(text: str) -> dict:
    text = text.strip()
    if not text:
        return {'language': 'english', 'lang_code': 'eng_Latn', 'method': 'fallback',
                'confidence': 0.0, 'is_code_mixed': False, 'is_romanized': False}

    has_latin     = bool(re.search(r'[a-zA-Z]', text))
    indic_chars   = len(re.findall(r'[\u0900-\u0D7F]', text))
    has_indic     = indic_chars >= 2
    is_code_mixed = has_latin and has_indic
    tokens        = text.split()

    # Stage 1: script detection for pure native-script text
    if not is_code_mixed:
        script_lang = detect_by_script(text)
        if script_lang:
            return {
                'language': script_lang,
                'lang_code': LANG_CODE.get(script_lang, 'eng_Latn'),
                'method': 'script',
                'confidence': 1.0,
                'is_code_mixed': False,
                'is_romanized': False,
            }

    # Stage 1b: code-mixed with dominant Indic script
    # Replace the entire Stage 1b block with this simpler version:
    if is_code_mixed and indic_chars >= 3:
        indic_only = re.sub(r'[a-zA-Z0-9\s\W]', '', text)
        if re.search(r'[\u0900-\u097F]', indic_only):
            # Devanagari dominant — default to Hindi
            # (Maithili/Bodo/Dogri in Devanagari are extremely rare in chat)
            return {
                'language': 'hindi',
                'lang_code': 'hin_Deva',
                'method': 'script',
                'confidence': 0.9,
                'is_code_mixed': True,
                'is_romanized': False,
            }
        else:
            script_lang = detect_by_script(indic_only)
            if script_lang:
                return {
                    'language': script_lang,
                    'lang_code': LANG_CODE.get(script_lang, 'eng_Latn'),
                    'method': 'script',
                    'confidence': 0.9,
                    'is_code_mixed': True,
                    'is_romanized': False,
                }


    # Pre-check: if no Indic chars, no code-mix, and no romanized hints found
    # → almost certainly English, don't bother calling IndicLID
    if not has_indic and not is_code_mixed:
        hint = detect_romanized_language(text)
        if not hint:
            return {
                'language': 'english',
                'lang_code': 'eng_Latn',
                'method': 'fallback',
                'confidence': 1.0,
                'is_code_mixed': False,
                'is_romanized': False,
            }

    # Stage 2: IndicLID (with romanized word-list pre-check built in)
    try:
        lid_lang, lid_conf, lid_code = detect_by_indiclid(text)
        is_romanized = '_Latn' in lid_code and lid_lang != 'english'
        min_conf = 0.7 if (len(tokens) <= 6 and not is_code_mixed) else 0.4
        if lid_conf >= min_conf:
            return {
                'language': lid_lang,
                'lang_code': lid_code,
                'method': 'indiclid',
                'confidence': lid_conf,
                'is_code_mixed': is_code_mixed,
                'is_romanized': is_romanized,
            }
    except Exception as e:
        print(f'  ⚠️ IndicLID error: {e}')

    return {
        'language': 'english',
        'lang_code': 'eng_Latn',
        'method': 'fallback',
        'confidence': 0.0,
        'is_code_mixed': is_code_mixed,
        'is_romanized': False,
    }


# Sanity check
test_inputs = [
    ('bhai server down hai kya?',       'Hinglish — romanized'),
    ('ಸರ್ವರ್ ಡೌನ್ ಆಗಿದೆ',              'Native Kannada'),
    ('ivathu office hogbeku illa',       'Kanglish — romanized'),
    ('can u check asap',                 'Pure English slang'),
    ('bhai यह काम नहीं कर रहा',         'Code-mixed Latin+Devanagari'),
    ('kal meeting 3pm irukku right?',    'Tanglish — romanized'),
    ('OTP नहीं आ रहा',                  'Native Devanagari (Hindi)'),
    ('aho, he kadhich honar nahi',       'Marathi — romanized'),
    ('oye kiddan paaji, sab theek?',     'Punjabi — romanized'),
    ('ayyo enthina ithra late?',         'Malayalam — romanized'),
]
print('\n--- Language Detection Sanity Check ---')
for text, description in test_inputs:
    r = detect_language(text)
    print(f'  [{description}]')
    print(f'  INPUT:  {text}')
    print(f'  → lang={r["language"]} | method={r["method"]} | conf={r["confidence"]:.2f} | romanized={r["is_romanized"]}')
    print()

Mounted at /content/drive
  ✅ FTN found at /content/drive/MyDrive/IndicLID_models/indiclid-ftn
  ✅ FTR found at /content/drive/MyDrive/IndicLID_models/indiclid-ftr
  ✅ BERT found at /content/drive/MyDrive/IndicLID_models/indiclid-bert
  ✅ Linked indiclid-ftn
  ✅ Linked indiclid-ftr
  ✅ Linked indiclid-bert


/content/IndicLID/Inference/ai4bharat/IndicLID.py:190: SyntaxWarning: invalid escape sequence '\|'
  special_char_pattern = re.compile('[@_!#$%^&*()<>?/\|}{~:]')
/content/IndicLID/Inference/ai4bharat/IndicLID.py:194: SyntaxWarning: invalid escape sequence '\s'
  spaces = len(re.findall('\s', input))


tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.75M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]


✅ IndicLID loaded successfully
✅ Script patterns built dynamically for: ['hindi', 'marathi', 'bengali', 'gujarati', 'punjabi', 'odia', 'tamil', 'telugu', 'kannada', 'malayalam']

--- Language Detection Sanity Check ---
  [Hinglish — romanized]
  INPUT:  bhai server down hai kya?
  → lang=hindi | method=indiclid | conf=1.00 | romanized=True

  [Native Kannada]
  INPUT:  ಸರ್ವರ್ ಡೌನ್ ಆಗಿದೆ
  → lang=kannada | method=script | conf=1.00 | romanized=False

  [Kanglish — romanized]
  INPUT:  ivathu office hogbeku illa
  → lang=kannada | method=indiclid | conf=1.00 | romanized=True

  [Pure English slang]
  INPUT:  can u check asap
  → lang=english | method=fallback | conf=1.00 | romanized=False

  [Code-mixed Latin+Devanagari]
  INPUT:  bhai यह काम नहीं कर रहा
  → lang=hindi | method=script | conf=0.90 | romanized=False

  [Tanglish — romanized]
  INPUT:  kal meeting 3pm irukku right?
  → lang=tamil | method=indiclid | conf=1.00 | romanized=True

  [Native Devanagari (Hindi)]
  INPUT:  OTP नह

In [ ]:
#DONT RUN
# See raw IndicLID output before any filtering
test_texts = [
    "bhai server down hai kya?",
    "ivathu office hogbeku illa",
    "OTP नहीं आ रहा",
    "kal meeting 3pm irukku right?",
]

for text in test_texts:
    result = indic_lid.batch_predict([text], batch_size=1)
    print(f"INPUT: {text}")
    print(f"RAW OUTPUT: {result}")
    print()

INPUT: bhai server down hai kya?
RAW OUTPUT: [('bhai server down hai kya?', 'ben_Latn', np.float32(0.96051663), 'IndicLID-FTR')]

INPUT: ivathu office hogbeku illa
RAW OUTPUT: [('ivathu office hogbeku illa', 'mal_Latn', np.float32(0.99987346), 'IndicLID-FTR')]

INPUT: OTP नहीं आ रहा
RAW OUTPUT: [('OTP नहीं आ रहा', 'mai_Deva', np.float32(1.00004), 'IndicLID-FTN')]

INPUT: kal meeting 3pm irukku right?
RAW OUTPUT: [('kal meeting 3pm irukku right?', 'tam_Latn', np.float32(0.9999255), 'IndicLID-FTR')]



---
## 3. Module 2 — Smart Routing (Heuristic Informality Scorer)

Decides whether a message needs normalization before translation.
Saves Mistral API calls for messages that are already clean.

**Scoring factors:**
- Known slang / internet abbreviation hits
- Short token ratio (abbreviation density)
- Character noise (repeated chars, excessive punctuation, all-caps)
- Code-mix flag from language detector
- Emoji presence

**Output:** `informality_score` in [0, 1]. Above threshold → normalize.

In [ ]:
SHORT_MSG_MAX_TOKENS = 2

SARCASM_EMOJIS = {"🙃", "👏", "🙄", "😒", "💀", "🤡"}
SARCASM_PHRASES = ["said no one ever", "just what i needed", "love how it",
                   "wow amazing", "fantastic,", "great, another", "nice. very nice",
                   "super smooth", "yeah right"]

def is_sarcastic(text: str) -> bool:
    t = text.lower()
    if any(e in text for e in SARCASM_EMOJIS):
        return True
    if any(p in t for p in SARCASM_PHRASES):
        return True
    return False

import re
from google.colab import userdata
from groq import Groq
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
groq_client = Groq(api_key=GROQ_API_KEY)
GROQ_MODEL = "llama-3.1-8b-instant"

SLANG_ABBREVS = {
    # Internet slang
    "idk", "idc", "imo", "imho", "irl", "fyi", "tbh", "tbf", "tbt",
    "lol", "lmao", "lmfao", "rofl", "omg", "omfg", "wtf", "wth",
    "brb", "bbl", "afk", "ttyl", "g2g", "gtg", "np", "nvm", "nm",
    "fr", "ngl", "smh", "fml", "smdh", "istg", "iirc",
    "af", "asf", "rn", "atm", "jk", "js",
    # Gen-Z slang
    "lowkey", "highkey", "slaps", "bussin", "mid", "slay", "rizz",
    "nocap", "cap", "based", "cringe", "vibe", "vibing",
    "yeet", "goat",
    # Indian English / office chat
    "eod", "eow", "asap", "ooo", "pto", "wfh", "wfo",
    "pls", "plz", "plzz", "thx", "thnx", "ty", "tyvm", "otp",
    "u", "r", "ur", "b4", "n", "2", "4",
    "msg", "msgs", "pw", "pwd", "acc", "diff", "prob",
    # Informal markers
    "bro", "bruh", "dude", "yaar", "bhai", "boss", "anna", "da",
    "yup", "nope", "nah", "yeah", "yep", "hmm", "ugh",
    "gonna", "wanna", "gotta", "kinda", "sorta", "lemme", "gimme",
    "ain't", "cuz", "coz", "cos", "tho", "thru",
}

CODEMIX_MARKERS = {
    "kya", "hai", "nahi", "aur", "bhi", "toh", "kal", "aaj",
    "swalpa", "maadi", "illa", "hogbeku", "ide", "aitu",
    "pannunga", "irukku", "poganuma", "varala", "panniten",
    "karo", "kar", "raha", "rahi", "gaya", "hoga", "tha", "thi",
"madana", "madidini", "madbeku", "aagtha", "aagala",
"baralla", "barthini", "call maadi",
"pannen", "pochu", "aachi", "pannuven", "irukka",
"bhej", "bhejo", "lelo", "dedo",
}

EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\U00002702-\U000027B0"
    "\U000024C2-\U0001F251"
    "]+",
    flags=re.UNICODE
)
def contains_figurative_language(text: str) -> bool:
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": "You are a linguistic classifier. Answer only 'yes' or 'no'. No explanation."},
            {"role": "user", "content": f"Does this message contain any idiom, figurative expression, sarcasm, or phrase that would translate incorrectly if done word-for-word?\n\nMessage: {text}"}
        ],
        temperature=0.0,
        max_tokens=5,
    )
    answer = response.choices[0].message.content.strip().lower()
    return answer.startswith("yes")

def contains_broken_english(text: str) -> bool:
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": "You are a linguistic classifier. Answer only 'yes' or 'no'. No explanation."},
            {"role": "user", "content": f"Is this message written in broken English, missing words, phonetic spelling, or incomplete grammar that needs to be cleaned up before translation?\n\nMessage: {text}"}
        ],
        temperature=0.0,
        max_tokens=5,
    )
    answer = response.choices[0].message.content.strip().lower()
    return answer.startswith("yes")

def compute_informality_score(text: str, is_code_mixed: bool = False) -> dict:
    """
    Computes an informality score in [0, 1].

    Returns:
        {
          'score': float,
          'needs_normalization': bool,
          'signals': dict
        }
    """
    THRESHOLD = 0.18  # lowered from 0.35 — more aggressive normalization

    # Strip punctuation before tokenizing so "af," matches "af"
    tokens = re.sub(r'[^\w\s]', ' ', text.lower()).split()
    if not tokens:
        return {"score": 0.0, "needs_normalization": False, "signals": {}}

    signals = {}

    # 1. Slang / abbreviation hits
    slang_hits = sum(1 for t in tokens if t in SLANG_ABBREVS)
    slang_ratio = slang_hits / len(tokens)
    # Bonus: if 2+ slang hits, boost the score
    if slang_hits >= 2:
        slang_ratio = min(slang_ratio * 2, 1.0)
    signals["slang_ratio"] = round(slang_ratio, 3)

    # 2. Code-mix markers
    codemix_hits = sum(1 for t in tokens if t in CODEMIX_MARKERS)
    signals["codemix_marker_ratio"] = round(codemix_hits / len(tokens), 3)

    # 3. Short token ratio (≤3 chars, alphabetic — likely abbreviations)
    short_tokens = [t for t in tokens if len(t) <= 3 and t.isalpha()]
    signals["short_token_ratio"] = round(len(short_tokens) / len(tokens), 3)

    # 4. Repeated chars ("brooooo", "whattttt")
    signals["repeated_chars"] = float(bool(re.search(r'(.)\1{2,}', text)))

    # 5. Excessive punctuation ("??", "!!!", "......")
    signals["excessive_punctuation"] = float(bool(re.search(r'[!?.]{2,}', text)))

    # 6. ALL CAPS words
    caps_words = [t for t in tokens if t.isupper() and len(t) > 2]
    signals["caps_ratio"] = round(len(caps_words) / len(tokens), 3)

    # 7. Emoji presence
    signals["has_emoji"] = float(bool(EMOJI_PATTERN.search(text)))

    # 8. Code-mix flag from language detector
    signals["is_code_mixed"] = float(is_code_mixed)

    # Weighted aggregation
    weights = {
        "slang_ratio":           0.35,
        "codemix_marker_ratio":  0.20,
        "short_token_ratio":     0.10,
        "repeated_chars":        0.20,
        "excessive_punctuation": 0.08,
        "caps_ratio":            0.08,
        "has_emoji":             0.05,
        "is_code_mixed":         0.02,
    }

    score = sum(weights[k] * min(signals[k], 1.0) for k in weights)
    score = round(min(score, 1.0), 4)
    has_figurative = contains_figurative_language(text)
    needs_norm = score >= THRESHOLD or has_figurative
    has_broken     = contains_broken_english(text)
    needs_norm     = score >= THRESHOLD or has_figurative or has_broken

    return {
        "score": score,
        "needs_normalization": needs_norm,
        "signals": signals
    }


# --- Sanity check ---
routing_tests = [
    "idk bro, this weather is wild af, raining cats and dogs fr",
    "Could you please provide an update on the ticket status?",
    "bhai server down hai kya? 😡",
    "The deployment has been completed successfully.",
    "BROOOOOOO why is it crashing again wtf???",
]
print("--- Smart Routing Sanity Check ---")
for t in routing_tests:
    r = compute_informality_score(t)
    print(f"  SCORE={r['score']:.3f} | NORMALIZE={r['needs_normalization']} | {t[:60]}")

--- Smart Routing Sanity Check ---
  SCORE=0.283 | NORMALIZE=True | idk bro, this weather is wild af, raining cats and dogs fr
  SCORE=0.040 | NORMALIZE=False | Could you please provide an update on the ticket status?
  SCORE=0.240 | NORMALIZE=True | bhai server down hai kya? 😡
  SCORE=0.033 | NORMALIZE=False | The deployment has been completed successfully.
  SCORE=0.387 | NORMALIZE=True | BROOOOOOO why is it crashing again wtf???


In [ ]:
#DOnt run
broken_tests = [
    "cant login, pw reset not working",
    "net slow so msg late",
    "app crash when open only",
    "i no understand why error coming",
    "why my account lock??",
]
for t in broken_tests:
    r = compute_informality_score(t)
    print(f"NORMALIZE={r['needs_normalization']} | {t}")

NORMALIZE=True | cant login, pw reset not working
NORMALIZE=True | net slow so msg late
NORMALIZE=True | app crash when open only
NORMALIZE=True | i no understand why error coming
NORMALIZE=True | why my account lock??


---
## 4. Module 3 — Context-Aware Normalization (Mistral)

**Upgrade from v1:** The normalization prompt now includes:
- **Conversation history** (last N messages) so references like "it", "that thing", "he said" resolve correctly
- **Speaker identity** labels so the model knows who said what in the thread

This prevents a common failure mode in chat translation where isolated normalization loses the conversational thread.

In [ ]:
import re
from collections import deque


# -------------------------------------------------------
# Conversation Context Manager
# Maintains a rolling window of the last N messages
# with speaker identity labels.
# -------------------------------------------------------
class ConversationContext:
    """
    Maintains a fixed-size rolling window of chat messages
    with speaker identity. Used to build context for normalization.

    Each entry: {'speaker': str, 'text': str}
    """

    def __init__(self, max_history: int = 5):
        self.max_history = max_history
        self.history = deque(maxlen=max_history)

    def add(self, speaker: str, text: str):
        """Add a message to the context window."""
        self.history.append({"speaker": speaker, "text": text})

    def format_for_prompt(self) -> str:
        """Format history as a labeled conversation block for the Mistral prompt."""
        if not self.history:
            return "(No prior conversation)"
        lines = [f"[{entry['speaker']}]: {entry['text']}" for entry in self.history]
        return "\n".join(lines)

    def reset(self):
        """Clear conversation — call this when starting a new chat session."""
        self.history.clear()

    def __len__(self):
        return len(self.history)


# -------------------------------------------------------
# Context-Aware Normalizer System Prompt
# Upgraded from v1: context window + speaker labels injected
# -------------------------------------------------------
CONTEXT_NORMALIZER_SYSTEM = """You are a semantic normalizer for a multilingual chat translation system.

You will receive:
1. CONVERSATION HISTORY: The last few messages with speaker labels, for context
2. CURRENT MESSAGE: The message to normalize

Your task: Rewrite ONLY the CURRENT MESSAGE into clear, grammatical English.

Hard rules:
- Rewrite ONLY the current message. Do not summarize or rewrite history.
- Never refuse. Never ask questions.
- Do NOT add new facts, times, places, names, or numbers not present in the input.
- Expand slang, abbreviations, idioms, and figurative expressions into standard English.
- Keep any non-English words exactly as they appear (do not translate them).
- Preserve question/statement form and sentiment (anger, sarcasm, urgency).
- ONLY use conversation history if the current message contains an explicit reference word like "it", "this", "that", "they", "the issue", "the meeting", "the same" etc.
- If the current message makes complete sense on its own WITHOUT the history, ignore the history entirely.
- NEVER pull in topics, objects, or subjects from history that are not directly referenced in the current message.
- Output ONLY the rewritten current message. No explanations, no labels, no preamble.
- SARCASM RULE: If the message is sarcastic or ironic (e.g. "wow amazing, server down again 👏", "love how it crashes", "said no one ever"), rewrite it to express the TRUE negative/frustrated meaning. E.g. "wow amazing service 👏" → "The service is terrible."
"""

def _clean_output(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^(assistant|Assistant)\s*[:\-]?\s*", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


print("✅ Figurative language detector ready")
def normalize_with_context(
    current_message: str,
    context: ConversationContext,
    speaker: str = "User",
    detected_language: str = "english"
) -> str:
    # Short-message guard — never normalise 1-2 token messages
    if len(current_message.strip().split()) <= SHORT_MSG_MAX_TOKENS:
        return current_message

    history_block = context.format_for_prompt()

    # Add vocabulary hints for code-mixed languages so Mistral normalizes correctly
    vocab_hints = {
        'kannada': "Kannada vocabulary hints: barthini=I'm coming, illa=no/not, hogbeku=need to go, maadi=do/please do, swalpa=a little, ide=there is, aitu=it's done, baralla=won't come, aagide=has happened, naale=tomorrow, ivathu=today",
        'hindi':   "Hindi vocabulary hints: hai=is, nahi=no/not, kya=what/is it, aur=and, bhi=also, toh=then, raha=going on, hoga=will be, kal=yesterday/tomorrow, aaj=today",
        'tamil':   "Tamil vocabulary hints: irukku=is/are, illa=no/not, pannunga=please do, varala=hasn't come, aaguthu=is happening, konjam=a little, romba=very, innaiku=today, naale=tomorrow",
        'telugu':  "Telugu vocabulary hints: undi=is/are, ledu=no/not, cheyyi=do it, avutundi=is happening, kaadu=no/not",
    }

    hint_line = ""
    if detected_language in vocab_hints:
        hint_line = f"\n\nVOCABULARY REFERENCE ({detected_language}):\n{vocab_hints[detected_language]}"

    user_prompt = f"""CONVERSATION HISTORY (for context only — do NOT rewrite this):
{history_block}

CURRENT MESSAGE (from {speaker} — rewrite this):
{current_message}{hint_line}"""

    response = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[
        {"role": "system", "content": CONTEXT_NORMALIZER_SYSTEM},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.0,
    max_tokens=200,
)
    return _clean_output(response.choices[0].message.content)

print("✅ Context-aware normalizer ready")
print("   ConversationContext window size: 5 messages")

✅ Figurative language detector ready
✅ Context-aware normalizer ready
   ConversationContext window size: 5 messages


---
## 5. Module 4 — BERTScore Evaluator

Measures semantic similarity between two texts using contextual embeddings.
Used in **evaluation mode only** — not called during live chat to keep latency low.

**Primary use:** Compare translation outputs from:
- Pipeline **without** normalization (raw → translate)
- Pipeline **with** normalization (raw → normalize → translate)

Both are back-translated to English for a fair comparison.

In [ ]:
from bert_score import score as bert_score_fn

def compute_bertscore(
    candidates: list[str],
    references: list[str],
    lang: str = "en"
) -> dict:
    """
    Computes BERTScore between candidate and reference strings.

    Args:
        candidates: List of generated/translated strings to evaluate.
        references: List of reference strings (ground truth or baseline).
        lang: Language code for BERTScore model selection.

    Returns:
        {
          'precision': list[float],
          'recall': list[float],
          'f1': list[float],
          'mean_f1': float
        }
    """
    P, R, F1 = bert_score_fn(
        candidates,
        references,
        lang=lang,
        verbose=False
    )
    return {
        "precision": P.tolist(),
        "recall":    R.tolist(),
        "f1":        F1.tolist(),
        "mean_f1":   round(float(F1.mean()), 4)
    }


def evaluate_pipeline_delta(
    test_cases: list[tuple[str, str]],
    context: ConversationContext,
    verbose: bool = True
) -> list[dict]:
    """
    Runs each test case through BOTH pipelines and compares via BERTScore:
      - v1 path: raw input → translate (no normalization)
      - v2 path: raw input → normalize (with context) → translate

    BERTScore is computed in English by comparing the normalized_english
    of v2 against the raw input, measuring how much meaning was preserved
    and clarified.

    Returns list of result dicts.
    """
    results = []

    for raw_text, target_lang in test_cases:
        tgt_code = LANG_CODE.get(target_lang.lower())
        if not tgt_code:
            continue

        # v1 path: translate raw input directly
        v1_translation = translate_with_indictrans([raw_text], tgt_lang=tgt_code)[0]

        # v2 path: normalize with context, then translate
        lang_info = detect_language(raw_text)
        routing  = compute_informality_score(raw_text, lang_info["is_code_mixed"])

        if routing["needs_normalization"]:
            normalized = normalize_with_context(raw_text, context, speaker="User")
        else:
            normalized = raw_text

        v2_translation = translate_with_indictrans([normalized], tgt_lang=tgt_code)[0]

        # BERTScore: normalized English vs raw input
        # (measures semantic preservation + improvement of normalization)
        bs = compute_bertscore([normalized], [raw_text], lang="en")

        result = {
            "input":              raw_text,
            "target_language":    target_lang,
            "normalized":         normalized,
            "normalization_skipped": not routing["needs_normalization"],
            "v1_translation":     v1_translation,
            "v2_translation":     v2_translation,
            "bertscore_f1":       bs["f1"][0],
            "informality_score":  routing["score"],
        }
        results.append(result)

        # Add this message to context for the next iteration
        context.add(speaker="User", text=raw_text)

        if verbose:
            print("-" * 80)
            print(f"INPUT:          {raw_text}")
            print(f"NORMALIZED:     {normalized}")
            print(f"V1 TRANSLATION: {v1_translation}")
            print(f"V2 TRANSLATION: {v2_translation}")
            print(f"BERTScore F1:   {bs['f1'][0]:.4f} | Informality: {routing['score']:.3f}")

    return results


print("✅ BERTScore evaluator ready (evaluation mode only)")

✅ BERTScore evaluator ready (evaluation mode only)


---
## 6. Master Function — `agentic_translate_v2()`

Orchestrates all four modules into a single call.
This is the drop-in replacement for `agentic_translate()` from v1.

In [ ]:
def agentic_translate_v2(
    user_text: str,
    target_language: str,
    context: 'ConversationContext',
    speaker: str = 'User',
    eval_mode: bool = False
) -> dict:
    """
    Full v2 pipeline:
      [1] Hybrid language detection (script regex → IndicLID → fallback)
      [2] Smart routing (heuristic informality score)
      [3] Context-aware normalization (Mistral + conversation history + speaker labels)
      [4] IndicTrans2 translation
      [5] BERTScore (eval_mode=True only)
    """
    target_language = target_language.lower()
    if target_language != 'english' and target_language not in LANG_CODE:
        raise ValueError(
            f"Unsupported target_language: '{target_language}'. "
            f"Choose from: {sorted(LANG_CODE.keys())}"
        )
    tgt_code = LANG_CODE.get(target_language, "eng_Latn")

    # ── [1] Language Detection ──────────────────────────────────────────────
    lang_info = detect_language(user_text)

    # ── [2] Smart Routing ───────────────────────────────────────────────────
    routing = compute_informality_score(user_text, lang_info['is_code_mixed'])

    # Normalization is triggered if message is informal OR romanized code-mixed
    # (romanized Indic text almost always benefits from normalization before translation)
    should_normalize = routing['needs_normalization'] or lang_info['is_romanized']

    # ── [3] Context-Aware Normalization ─────────────────────────────────────
    # Never normalise 1-2 token messages regardless of routing score
    if len(user_text.strip().split()) <= SHORT_MSG_MAX_TOKENS:
        should_normalize = False

    if should_normalize:
        normalized = normalize_with_context(user_text, context, speaker=speaker, detected_language=lang_info['language'])
        normalization_skipped = False
    else:
        normalized = user_text
        normalization_skipped = True
   # ── [4] Translation ──────────────────────────────────────────────────────
    src_is_indic = lang_info['language'] != 'english' and not lang_info['is_romanized']
    tgt_is_indic = target_language in LANG_CODE
    same_language = lang_info['language'] == target_language

    if target_language == 'english':
      src_code = INDIC_TO_ENG_CODE.get(lang_info['language'])
      if src_code:
          translation = translate_indic_to_english([normalized], src_lang=src_code)[0]
      else:
          translation = normalized
      english_pivot = None

    elif src_is_indic and tgt_is_indic and not same_language:
        # Indic → Indic: use English pivot
        translation, english_pivot = translate_indic_to_indic(
            normalized,
            src_language=lang_info['language'],
            tgt_language=target_language
        )
    else:
        # English/Romanized → Indic: direct translation
        english_pivot = None
        translation = translate_with_indictrans(
            [normalized],
            src_lang='eng_Latn',
            tgt_lang=tgt_code
        )[0]

    # ── [5] BERTScore (evaluation mode only) ────────────────────────────────
    bertscore_f1 = None
    if eval_mode:
        bs = compute_bertscore([normalized], [user_text], lang='en')
        bertscore_f1 = round(bs['f1'][0], 4)

    # Update context window with the raw message
    context.add(speaker=speaker, text=user_text)

    return {
        'input':                 user_text,
        'speaker':               speaker,
        # Language detection
        'detected_language':     lang_info['language'],
        'detected_lang_code':    lang_info['lang_code'],
        'detection_method':      lang_info['method'],
        'detection_confidence':  lang_info['confidence'],
        'is_code_mixed':         lang_info['is_code_mixed'],
        'is_romanized':          lang_info['is_romanized'],
        # Smart routing
        'informality_score':     routing['score'],
        'normalization_skipped': normalization_skipped,
        'routing_signals':       routing['signals'],
        # Normalization
        'normalized_english':    normalized,
        'context_messages_used': len(context) - 1,
        # Translation
        'target_language':       target_language,
        'translation':           translation,
        'english_pivot':         english_pivot,
        # Evaluation
        'bertscore_f1':          bertscore_f1,
    }


print('✅ agentic_translate_v2() ready')


✅ agentic_translate_v2() ready


In [ ]:
##TEST
ctx = ConversationContext()

test_cases = [
    ("சர்வர் டவுன் ஆகிவிட்டது", "tamil",    "hindi"),
    ("ಸರ್ವರ್ ಡೌನ್ ಆಗಿದೆ",        "kannada",  "bengali"),
    ("সার্ভার ডাউন হয়েছে",         "bengali",  "tamil"),
    ("സർവർ ഡൗൺ ആയി",             "malayalam","hindi"),
    ("ਸਰਵਰ ਡਾਊਨ ਹੋ ਗਿਆ",           "punjabi",  "telugu"),
]

for text, src, tgt in test_cases:
    out = agentic_translate_v2(text, tgt, ctx, "User")
    print(f"INPUT [{src} → {tgt}]: {text}")
    if out.get('english_pivot'):
        print(f"  English pivot: {out['english_pivot']}")
    print(f"  Translation:   {out['translation']}")
    print()

INPUT [tamil → hindi]: சர்வர் டவுன் ஆகிவிட்டது
  English pivot: The server is down.
  Translation:   सर्वर डाउन है।

INPUT [kannada → bengali]: ಸರ್ವರ್ ಡೌನ್ ಆಗಿದೆ
  English pivot: The server is down.
  Translation:   সার্ভার ডাউন হয়ে গেছে।

INPUT [bengali → tamil]: সার্ভার ডাউন হয়েছে
  English pivot: The server is down.
  Translation:   சேவையகம் செயலிழந்துள்ளது.

INPUT [malayalam → hindi]: സർവർ ഡൗൺ ആയി
  English pivot: The server is down.
  Translation:   सर्वर डाउन है।

INPUT [punjabi → telugu]: ਸਰਵਰ ਡਾਊਨ ਹੋ ਗਿਆ
  English pivot: The server is down.
  Translation:   సర్వర్ డౌన్ అయింది.



---
## 7. Quick Demo — Context-Aware Translation in Action

This shows the key upgrade: messages reference each other, and the normalizer resolves those references correctly.

In [ ]:
# Start a fresh conversation context
ctx = ConversationContext(max_history=5)

# Simulate a short conversation thread
convo = [
    ("User A", "bhai server down hai kya?",              "hindi"),
    ("User B", "haan yaar, main dekh raha hu",           "hindi"),
    ("User A", "ok bro let me know asap, super urgent",  "hindi"),
    ("User B", "done, it's back up",                     "hindi"),
    ("User A", "ty bhai, wont happen again right?",      "hindi"),
]

print("=" * 90)
print("CONTEXT-AWARE TRANSLATION DEMO — Conversation Thread")
print("=" * 90)

for speaker, message, lang in convo:
    result = agentic_translate_v2(
        user_text=message,
        target_language=lang,
        context=ctx,
        speaker=speaker
    )
    print(f"\n[{speaker}]")
    print(f"  RAW:        {result['input']}")
    print(f"  NORMALIZED: {result['normalized_english']}  "
          f"{'(skipped)' if result['normalization_skipped'] else ''}")
    print(f"  TRANSLATED: {result['translation']}")
    print(f"  detected={result['detected_language']} | "
          f"informality={result['informality_score']:.3f} | "
          f"context_msgs={result['context_messages_used']}")

CONTEXT-AWARE TRANSLATION DEMO — Conversation Thread

[User A]
  RAW:        bhai server down hai kya?
  NORMALIZED: Is the server down?  
  TRANSLATED: सर्वर डाउन है?
  detected=hindi | informality=0.190 | context_msgs=0

[User B]
  RAW:        haan yaar, main dekh raha hu
  NORMALIZED: The server is down, I'm checking.  
  TRANSLATED: सर्वर डाउन है, मैं जाँच कर रहा हूँ।
  detected=hindi | informality=0.109 | context_msgs=1

[User A]
  RAW:        ok bro let me know asap, super urgent
  NORMALIZED: The server is down, please let me know as soon as possible, it's extremely urgent.  
  TRANSLATED: सर्वर डाउन है, कृपया मुझे जल्द से जल्द बताएँ, यह बेहद जरूरी है।
  detected=english | informality=0.225 | context_msgs=2

[User B]
  RAW:        done, it's back up
  NORMALIZED: The server is back up now.  
  TRANSLATED: सर्वर अब वापस आ गया है।
  detected=english | informality=0.060 | context_msgs=3

[User A]
  RAW:        ty bhai, wont happen again right?
  NORMALIZED: The server will not go d

---
## 8. Stress Tests — v2 Pipeline on All 120+ Cases

In [ ]:
import random
import time

STRESS_TESTS_120 = [

    # ── 1. SHORT-MESSAGE GUARD (should pass through unchanged) ──────────────
    ("ok",                                      "hindi"),
    ("yes",                                     "kannada"),
    ("done",                                    "tamil"),

    # ── 2. HINGLISH ─────────────────────────────────────────────────────────
    ("bhai server down hai kya?",               "hindi"),
    ("yaar OTP nahi aa raha, kya karu?",        "hindi"),
    ("pls thoda jaldi karo, urgent hai",        "hindi"),
    ("kal 3 baje meeting fix hai na?",          "hindi"),
    ("thanks bhai, but issue abhi bhi hai",     "hindi"),

    # ── 3. KANGLISH ─────────────────────────────────────────────────────────
    ("bro swalpa adjust maadi, urgent ide",     "kannada"),
    ("naale meeting reschedule madana?",        "kannada"),
    ("kelsa complete madidini, pls review",     "kannada"),   # previously broken
    ("payment aagide but receipt baralla",      "kannada"),   # previously broken
    ("naan barthini, but swalpa late",          "kannada"),

    # ── 4. TANGLISH ─────────────────────────────────────────────────────────
    ("bro konjam wait pannunga, net slow",      "tamil"),
    ("app open panna crash aaguthu",            "tamil"),
    ("ticket raise panniten, reply varala",     "tamil"),     # previously broken
    ("enaku OTP varala, help pannunga",         "tamil"),
    ("server down ah? romba worst",             "tamil"),

    # ── 5. NATIVE SCRIPT — Hindi ────────────────────────────────────────────
    ("सर्वर डाउन हो गया है",                   "kannada"),
    ("मेरा भुगतान अटक गया है",                 "tamil"),
    ("कृपया जल्दी करें",                        "telugu"),

    # ── 6. NATIVE SCRIPT — Kannada ──────────────────────────────────────────
    ("ಸರ್ವರ್ ಡೌನ್ ಆಗಿದೆ",                      "hindi"),
    ("ಪಾವತಿ ಮಾಡಲಾಗಿದೆ",                        "tamil"),

    # ── 7. NATIVE SCRIPT — Tamil ────────────────────────────────────────────
    ("சர்வர் டவுன் ஆகிவிட்டது",                "hindi"),
    ("பணம் திரும்பவில்லை",                     "kannada"),

    # ── 8. IDIOMS ───────────────────────────────────────────────────────────
    ("break a leg for your interview",          "hindi"),
    ("idk man, it's raining cats and dogs",     "kannada"),
    ("that's a piece of cake",                  "tamil"),
    ("costs an arm and a leg",                  "telugu"),

    # ── 9. GEN-Z / INTERNET SLANG ───────────────────────────────────────────
    ("ngl that was kinda mid",                  "tamil"),
    ("fr this app slaps",                       "telugu"),
    ("lowkey I'm done with this",               "marathi"),
    ("no cap, that's impressive",               "telugu"),

    # ── 10. OFFICE ABBREVIATIONS ────────────────────────────────────────────
    ("pls share update by eod",                 "hindi"),
    ("cant login, pw reset not working",        "kannada"),
    ("plz do needful asap",                     "hindi"),

    # ── 11. SARCASM (previously failing — testing fix) ──────────────────────
    ("nice. very nice. nothing works.",         "kannada"),
    ("sure, take your time... not urgent 🙃",   "telugu"),
    ("wow amazing service, server down again 👏","hindi"),
    ("fantastic, payment failed third time",    "bengali"),

]

random.seed(42)
random.shuffle(STRESS_TESTS_120)
print(f"Total test cases: {len(STRESS_TESTS_120)}")


def run_stress_tests_v2(
    tests,
    batch_size: int = 10,
    sleep_s: float = 1.0,
    eval_mode: bool = False
) -> list[dict]:
    """
    Runs all test cases through agentic_translate_v2.
    Each batch uses a fresh ConversationContext so batches
    don't bleed context into each other.
    """
    all_results = []

    for i in range(0, len(tests), batch_size):
        batch = tests[i : i + batch_size]
        batch_ctx = ConversationContext(max_history=5)

        print("\n" + "#" * 100)
        print(f"Batch {i // batch_size + 1} | cases {i + 1}–{min(i + batch_size, len(tests))}")
        print("#" * 100)

        for text, lang in batch:
            try:
                out = agentic_translate_v2(
                    user_text=text,
                    target_language=lang,
                    context=batch_ctx,
                    speaker="User",
                    eval_mode=eval_mode
                )
                all_results.append(out)
                skip_tag = " [NORM SKIPPED]" if out["normalization_skipped"] else ""
                print("-" * 90)
                print(f"INPUT:      {out['input']}")
                print(f"NORMALIZED: {out['normalized_english']}{skip_tag}")
                print(f"LANG:       {out['target_language']} | "
                      f"detected={out['detected_language']} ({out['detection_method']}) | "
                      f"informality={out['informality_score']:.3f}")
                print(f"TRANSLATED: {out['translation']}")
                if eval_mode and out["bertscore_f1"] is not None:
                    print(f"BERTScore F1: {out['bertscore_f1']:.4f}")
            except Exception as e:
                all_results.append({"input": text, "target_language": lang, "error": str(e)})
                print(f"-" * 90)
                print(f"INPUT: {text}")
                print(f"ERROR: {e}")

        time.sleep(sleep_s)

    return all_results


# Run — set eval_mode=True to also compute BERTScores (slower)
stress_results_v2 = run_stress_tests_v2(
    STRESS_TESTS_120,
    batch_size=8,
    sleep_s=0.6,
    eval_mode=False   # flip to True for full evaluation run
)

print(f"\n✅ Done. {len(stress_results_v2)} results stored in: stress_results_v2")

Total test cases: 40

####################################################################################################
Batch 1 | cases 1–8
####################################################################################################
------------------------------------------------------------------------------------------
INPUT:      naale meeting reschedule madana?
NORMALIZED: The meeting is rescheduled for tomorrow.
LANG:       kannada | detected=kannada (indiclid) | informality=0.050
TRANSLATED: ಸಭೆಯನ್ನು ನಾಳೆಗೆ ಮರು ನಿಗದಿಪಡಿಸಲಾಗಿದೆ.
------------------------------------------------------------------------------------------
INPUT:      bhai server down hai kya?
NORMALIZED: Is the server down?
LANG:       hindi | detected=hindi (indiclid) | informality=0.190
TRANSLATED: सर्वर डाउन है?
------------------------------------------------------------------------------------------
INPUT:      kelsa complete madidini, pls review
NORMALIZED: The task is complete madidini, please revie

---
## 9. Results Summary & Pipeline Comparison

Summarizes smart routing efficiency and BERTScore results (if eval_mode was True).

In [ ]:
import statistics

successful = [r for r in stress_results_v2 if "error" not in r]
errors     = [r for r in stress_results_v2 if "error" in r]

normalized_count = sum(1 for r in successful if not r["normalization_skipped"])
skipped_count    = sum(1 for r in successful if r["normalization_skipped"])

codemix_count    = sum(1 for r in successful if r["is_code_mixed"])

detection_methods = {}
for r in successful:
    m = r["detection_method"]
    detection_methods[m] = detection_methods.get(m, 0) + 1

informality_scores = [r["informality_score"] for r in successful]

print("=" * 70)
print("📊 V2 PIPELINE SUMMARY")
print("=" * 70)
print(f"  Total test cases:         {len(stress_results_v2)}")
print(f"  Successful:               {len(successful)}")
print(f"  Errors:                   {len(errors)}")
print()
print(f"── Smart Routing ──")
print(f"  Normalized (Mistral called):  {normalized_count} ({100*normalized_count/len(successful):.1f}%)")
print(f"  Skipped (already clean):      {skipped_count} ({100*skipped_count/len(successful):.1f}%)")
print(f"  API calls saved vs v1:        {skipped_count} / {len(successful)}")
print()
print(f"── Language Detection ──")
for method, count in sorted(detection_methods.items(), key=lambda x: -x[1]):
    print(f"  {method:<12}: {count} messages")
print(f"  Code-mixed detected:          {codemix_count}")
print()
print(f"── Informality Scores ──")
print(f"  Mean:   {statistics.mean(informality_scores):.3f}")
print(f"  Median: {statistics.median(informality_scores):.3f}")
print(f"  Min:    {min(informality_scores):.3f}")
print(f"  Max:    {max(informality_scores):.3f}")

# BERTScore summary (only if eval_mode was True)
bs_scores = [r["bertscore_f1"] for r in successful if r.get("bertscore_f1") is not None]
if bs_scores:
    print()
    print(f"── BERTScore F1 (Normalized vs Raw) ──")
    print(f"  Mean:   {statistics.mean(bs_scores):.4f}")
    print(f"  Median: {statistics.median(bs_scores):.4f}")
    print(f"  Min:    {min(bs_scores):.4f}")
    print(f"  Max:    {max(bs_scores):.4f}")
else:
    print()
    print("  ℹ️  BERTScores not computed — re-run with eval_mode=True for full evaluation")

print("=" * 70)

📊 V2 PIPELINE SUMMARY
  Total test cases:         40
  Successful:               40
  Errors:                   0

── Smart Routing ──
  Normalized (Mistral called):  35 (87.5%)
  Skipped (already clean):      5 (12.5%)
  API calls saved vs v1:        5 / 40

── Language Detection ──
  fallback    : 19 messages
  indiclid    : 17 messages
  script      : 4 messages
  Code-mixed detected:          0

── Informality Scores ──
  Mean:   0.134
  Median: 0.100
  Min:    0.000
  Max:    0.410

  ℹ️  BERTScores not computed — re-run with eval_mode=True for full evaluation


In [ ]:
# ── Evaluation Dataset ─────────────────────────────────────────────────────
# 60 sentences across 14 language/style categories
# Each tuple: (input_text, target_language, human_reference_translation)
# Reference translations provided for Hindi target only for now
# For Tamil/Telugu targets — BERTScore against v2 output as pseudo-reference

EVAL_DATASET = [
    # --- Hinglish romanized (8) ---
    ("bhai server down hai kya?",               "hindi",   "सर्वर डाउन है, भाई?"),
    ("yaar server firse down ho gaya",           "hindi",   "यार, सर्वर फिर से डाउन हो गया है।"),
    ("pls thoda jaldi karo, urgent hai",         "hindi",   "कृपया इसे थोड़ा जल्दी करें, यह जरूरी है।"),
    ("mera payment stuck hai, check karo",       "hindi",   "मेरा भुगतान अटक गया है, कृपया जाँच करें।"),
    ("aaj call kar sakte ho kya?",               "hindi",   "क्या आप आज फोन कर सकते हैं?"),
    ("thanks bhai, but issue abhi bhi hai",      "hindi",   "धन्यवाद भाई, लेकिन समस्या अभी भी है।"),
    ("boss ko message bhej du kya?",             "hindi",   "क्या मुझे बॉस को संदेश भेजना चाहिए?"),
    ("OTP nahi aa raha, kya karu?",              "hindi",   "OTP नहीं आ रहा है, मुझे क्या करना चाहिए?"),

    # --- Kanglish romanized (6) ---
    ("naale meeting reschedule madana?",         "kannada", "ನಾಳೆಯ ಸಭೆಯನ್ನು ಮರು ನಿಗದಿಪಡಿಸಬೇಕೇ?"),
    ("bro swalpa adjust maadi, urgent ide",      "kannada", "ದಯವಿಟ್ಟು ಸ್ವಲ್ಪ ಹೊಂದಾಣಿಕೆ ಮಾಡಿ, ತುರ್ತು ಇದೆ।"),
    ("payment aagide but receipt baralla",       "kannada", "ಪಾವತಿ ಆಗಿದೆ ಆದರೆ ರಸೀದಿ ಬಂದಿಲ್ಲ।"),
    ("ninna night app hang aitu",                "kannada", "ನಿನ್ನೆ ರಾತ್ರಿ ಅಪ್ಲಿಕೇಶನ್ ಸ್ಥಗಿತಗೊಂಡಿತು।"),
    ("call maadi, important",                    "kannada", "ದಯವಿಟ್ಟು ಕರೆ ಮಾಡಿ, ಮುಖ್ಯವಾಗಿದೆ।"),
    ("kelsa complete madidini, pls review",      "kannada", "ಕೆಲಸ ಪೂರ್ಣಗೊಳಿಸಿದ್ದೇನೆ, ದಯವಿಟ್ಟು ಪರಿಶೀಲಿಸಿ।"),

    # --- Tanglish romanized (6) ---
    ("kal meeting 3pm irukku right?",            "tamil",   "கூட்டம் நாளை மதியம் 3 மணிக்கு இருக்கிறது, சரியா?"),
    ("bro konjam wait pannunga, net slow",       "tamil",   "கொஞ்சம் காத்திருங்கள், நெட் ஸ்லோ ஆக இருக்கிறது।"),
    ("app open panna crash aaguthu",             "tamil",   "ஆப் திறக்கும்போது க்ராஷ் ஆகிறது।"),
    ("enaku OTP varala, help pannunga",          "tamil",   "எனக்கு OTP வரவில்லை, உதவுங்கள்।"),
    ("ticket raise panniten, reply varala",      "tamil",   "டிக்கெட் போட்டேன், பதில் வரவில்லை।"),
    ("server down ah? romba worst",              "tamil",   "சர்வர் டவுன் ஆச்சா? மிகவும் மோசமாக இருக்கிறது।"),

    # --- Native Hindi Devanagari (5) ---
    ("सर्वर डाउन हो गया है",                    "hindi",   "सर्वर डाउन हो गया है।"),
    ("OTP नहीं आ रहा",                          "hindi",   "OTP नहीं आ रहा है।"),
    ("मेरा भुगतान अटक गया है",                  "hindi",   "मेरा भुगतान अटक गया है।"),
    ("कृपया जल्दी करें",                         "hindi",   "कृपया जल्दी करें।"),
    ("बैठक कल है",                               "hindi",   "बैठक कल है।"),

    # --- Native Kannada script (5) ---
    ("ಸರ್ವರ್ ಡೌನ್ ಆಗಿದೆ",                       "kannada", "ಸರ್ವರ್ ಡೌನ್ ಆಗಿದೆ।"),
    ("ಅಪ್ಲಿಕೇಶನ್ ತೆರೆಯಲು ಸಾಧ್ಯವಾಗುತ್ತಿಲ್ಲ",      "kannada", "ಅಪ್ಲಿಕೇಶನ್ ತೆರೆಯಲು ಸಾಧ್ಯವಾಗುತ್ತಿಲ್ಲ।"),
    ("ಪಾವತಿ ಮಾಡಲಾಗಿದೆ",                          "kannada", "ಪಾವತಿ ಮಾಡಲಾಗಿದೆ।"),
    ("ದಯವಿಟ್ಟು ಸಹಾಯ ಮಾಡಿ",                       "kannada", "ದಯವಿಟ್ಟು ಸಹಾಯ ಮಾಡಿ।"),
    ("ಸಭೆ ನಾಳೆ ಇದೆ",                             "kannada", "ಸಭೆ ನಾಳೆ ಇದೆ।"),

    # --- Native Tamil script (5) ---
    ("சர்வர் டவுன் ஆகிவிட்டது",                  "tamil",   "சர்வர் டவுன் ஆகிவிட்டது।"),
    ("பணம் திரும்பவில்லை",                        "tamil",   "பணம் திரும்பவில்லை।"),
    ("OTP வரவில்லை",                              "tamil",   "OTP வரவில்லை।"),
    ("தயவுசெய்து உதவுங்கள்",                      "tamil",   "தயவுசெய்து உதவுங்கள்।"),
    ("கூட்டம் நாளை இருக்கிறது",                   "tamil",   "கூட்டம் நாளை இருக்கிறது।"),

    # --- Native Telugu script (4) ---
    ("సర్వర్ డౌన్ అయింది",                        "telugu",  "సర్వర్ డౌన్ అయింది।"),
    ("చెల్లింపు నిలిచిపోయింది",                   "telugu",  "చెల్లింపు నిలిచిపోయింది।"),
    ("దయచేసి సహాయం చేయండి",                      "telugu",  "దయచేసి సహాయం చేయండి।"),
    ("రేపు సమావేశం ఉంది",                         "telugu",  "రేపు సమావేశం ఉంది।"),

    # --- Native Bengali script (4) ---
    ("সার্ভার ডাউন হয়েছে",                        "bengali", "সার্ভার ডাউন হয়েছে।"),
    ("পেমেন্ট আটকে গেছে",                         "bengali", "পেমেন্ট আটকে গেছে।"),
    ("OTP আসেনি",                                 "bengali", "OTP আসেনি।"),
    ("দয়া করে সাহায্য করুন",                      "bengali", "দয়া করে সাহায্য করুন।"),

    # --- Native Marathi script (3) ---
    ("सर्व्हर डाउन झाला आहे",                    "marathi", "सर्व्हर डाउन झाला आहे।"),
    ("पेमेंट अडकले आहे",                          "marathi", "पेमेंट अडकले आहे।"),
    ("उद्या बैठक आहे",                            "marathi", "उद्या बैठक आहे।"),

    # --- Native Malayalam script (2) ---
    ("സർവർ ഡൗൺ ആയി",                            "malayalam","സർവർ ഡൗൺ ആയി।"),
    ("പേയ്മെന്റ് കുടുങ്ങി",                       "malayalam","പേയ്മെന്റ് കുടുങ്ങി।"),

    # --- Native Gujarati script (2) ---
    ("સર્વર ડાઉન થઈ ગયું",                       "gujarati", "સર્વર ડાઉન થઈ ગયું।"),
    ("ચુકવણી અટકી ગઈ છે",                        "gujarati", "ચુકવણી અટકી ગઈ છે।"),

    # --- Native Punjabi script (2) ---
    ("ਸਰਵਰ ਡਾਊਨ ਹੋ ਗਿਆ",                         "punjabi", "ਸਰਵਰ ਡਾਊਨ ਹੋ ਗਿਆ।"),
    ("ਭੁਗਤਾਨ ਅਟਕ ਗਿਆ ਹੈ",                        "punjabi", "ਭੁਗਤਾਨ ਅਟਕ ਗਿਆ ਹੈ।"),

    # --- Native Odia script (2) ---
    ("ସର୍ଭର ଡାଉନ ହୋଇଛି",                         "odia",    "ସର୍ଭର ଡାଉନ ହୋଇଛି।"),
    ("ପେମେଣ୍ଟ ଅଟକି ଯାଇଛି",                        "odia",    "ପେମେଣ୍ଟ ଅଟକି ଯାଇଛି।"),

    # --- Informal English slang/idioms (6) ---
    ("idk man, it's raining cats and dogs",      "hindi",   "मुझे नहीं पता, बहुत तेज़ बारिश हो रही है।"),
    ("bruh this is wild af",                     "hindi",   "यार, यह बिल्कुल बेतुका है।"),
    ("ngl that was kinda mid",                   "hindi",   "सच कहूँ तो, वह कुछ खास नहीं था।"),
    ("cant login, pw reset not working",         "hindi",   "लॉगिन नहीं हो रहा, पासवर्ड रीसेट भी काम नहीं कर रहा।"),
    ("pls share update by eod",                  "hindi",   "कृपया आज दिन के अंत तक अपडेट शेयर करें।"),
    ("BROOOOOOO why is it crashing again wtf???","hindi",   "यार, यह फिर से क्रैश क्यों हो रहा है?"),
]

print(f"✅ Evaluation dataset ready — {len(EVAL_DATASET)} sentences")

# Check all target languages are supported
unsupported = set(lang for _, lang, _ in EVAL_DATASET) - set(LANG_CODE.keys())
if unsupported:
    print(f"⚠️ Unsupported languages: {unsupported}")
else:
    print(f"✅ All target languages supported")

✅ Evaluation dataset ready — 60 sentences
✅ All target languages supported


In [ ]:
# Convert FLORES sentences into your EVAL_DATASET tuple format
# Your format is: (input_text, target_language, reference_translation)

flores_eval_entries = []

for s in flores_sentences:
    flores_eval_entries.append((
        s["input"],
        s["tgt_lang"],
        s["reference"]
    ))

print(f"✅ Converted {len(flores_eval_entries)} sentences to eval format")
print(f"\nPreview:")
for entry in flores_eval_entries[:3]:
    print(f"  Input:  {entry[0][:60]}...")
    print(f"  Target: {entry[1]}")
    print(f"  Ref:    {entry[2][:60]}...")
    print()

✅ Converted 30 sentences to eval format

Preview:
  Input:  """எங்களிடம் இப்போது  4-மாத-வயதுடைய எலி ஒன்று உள்ளது, முன்னர...
  Target: hindi
  Ref:    उन्होंने कहा “कि अब हमारे पास 4 महीने उम्र वाले चूहे हैं जिन...

  Input:  ஹாலிஃபாக்ஸில் உள்ள டல்ஹெளசி பல்கலைக்கழகத்தின் மருத்துவப் பேர...
  Target: hindi
  Ref:    डॉ. एहुड उर, नोवा स्कोटिया के हैलिफ़ैक्स में डलहौज़ी विश्ववि...

  Input:  பிற வல்லுனர்கள் போலவே அவரும், ஏற்கனவே டைப் 1 இரத்த சர்க்கரை ...
  Target: hindi
  Ref:    कुछ अन्य विशेषज्ञों की तरह, उन्हें इस बात पर संदेह है कि क्य...



In [ ]:
MY_HINGLISH = [
    ("bhai aaj meeting cancel ho gayi kya?",                "hindi", "क्या आज की बैठक रद्द हो गई?"),
    ("yaar password bhool gaya, kya karu",                  "hindi", "मैं अपना पासवर्ड भूल गया, मुझे क्या करना चाहिए?"),
    ("kal tak submit karna hai, tension ho rahi hai",        "hindi", "कल तक जमा करना है, चिंता हो रही है।"),
    ("bhai net bahut slow hai aaj",                         "hindi", "आज इंटरनेट बहुत धीमा है।"),
    ("yaar OTP aa gaya, ab login ho gaya",                  "hindi", "एकबारगी पासवर्ड आ गया, अब प्रवेश हो गया।"),
    ("boss ne kaha eod tak bhej do report",                 "hindi", "प्रबंधक ने कहा कि आज कार्यदिवस समाप्त होने से पहले रिपोर्ट भेज दो।"),
    ("bhai app update karna hai kya?",                      "hindi", "क्या अनुप्रयोग को अद्यतन करना आवश्यक है?"),
    ("yaar call drop ho raha hai baar baar",                "hindi", "बार-बार कॉल कट रही है।"),
    ("payment ho gayi bhai, screenshot bhej raha hu",       "hindi", "भुगतान हो गया है, मैं स्क्रीनशॉट भेज रहा हूं।"),
    ("aaj wfh hai kya office mein?",                        "hindi", "क्या आज घर से काम करना है?"),
    ("bhai login nahi ho raha, password reset karna hai",   "hindi", "प्रवेश नहीं हो रहा, पासवर्ड पुनः स्थापित करना होगा।"),
    ("yaar deadline aaj hai, thoda help karo",              "hindi", "आज अंतिम तिथि है, कृपया थोड़ी सहायता करो।"),
    ("meeting mein join nahi ho pa raha, link bhejo",       "hindi", "बैठक में शामिल नहीं हो पा रहा, कृपया लिंक भेजें।"),
    ("bhai refund kab tak aayega?",                         "hindi", "धनवापसी कब तक मिलेगी?"),
    ("yaar system hang ho gaya, restart karna padega",      "hindi", "तंत्र प्रतिसाद देना बंद हो गया, पुनः आरंभ करना होगा।"),
    ("aaj presentation hai, nervous hu thoda",              "hindi", "आज प्रस्तुति है, थोड़ी घबराहट हो रही है।"),
    ("bhai file upload nahi ho rahi, size issue hai kya?",  "hindi", "फ़ाइल अपलोड नहीं हो रही, क्या आकार की समस्या है?"),
    ("yaar interview kal hai, wish me luck",                "hindi", "कल साक्षात्कार है, मुझे शुभकामनाएं दो।"),
    ("bhai account block ho gaya, kya karu?",               "hindi", "खाता अवरुद्ध हो गया है, मुझे क्या करना चाहिए?"),
    ("thoda baad baat karte hain, abhi busy hu",            "hindi", "थोड़ी देर बाद बात करते हैं, अभी व्यस्त हूं।"),
    ("yaar discount code kaam nahi kar raha",               "hindi", "छूट कोड काम नहीं कर रहा।"),
    ("bhai kal chutti hai kya office mein?",                "hindi", "क्या कल कार्यालय में अवकाश है?"),
]

MY_KANGLISH = [
    ("bro naale office bartheera?",                         "kannada", "ನಾಳೆ ಕಚೇರಿಗೆ ಬರುತ್ತೀರಾ?"),
    ("yen da idu, app again crash aagide",                  "kannada", "ಏನಿದು, ಅನ್ವಯ ಮತ್ತೆ ಕಾರ್ಯನಿಲ್ಲಿಸಿದೆ."),
    ("meeting link kalsilla, please kalshu",                "kannada", "ಸಭೆಯ ಕೊಂಡಿ ಕಳಿಸಿಲ್ಲ, ದಯವಿಟ್ಟು ಕಳಿಸಿ."),
    ("bro ivathu deadline ide, help madu",                  "kannada", "ಇಂದು ಗಡುವು ಇದೆ, ದಯವಿಟ್ಟು ಸಹಾಯ ಮಾಡಿ."),
    ("net illde meeting join madbeku",                      "kannada", "ಅಂತರ್ಜಾಲವಿಲ್ಲದೆ ಸಭೆಯಲ್ಲಿ ಭಾಗವಹಿಸಬೇಕಾಗಿದೆ."),
    ("bro account block aagide, yenu madodu?",              "kannada", "ಖಾತೆ ನಿರ್ಬಂಧಿಸಲ್ಪಟ್ಟಿದೆ, ಏನು ಮಾಡಬೇಕು?"),
    ("ivathu presentation ide, nervous aagide",             "kannada", "ಇಂದು ಪ್ರಸ್ತುತಿ ಇದೆ, ಆತಂಕವಾಗಿದೆ."),
    ("refund yaavaga baratte bro?",                         "kannada", "ಹಣ ವಾಪಸಾತಿ ಯಾವಾಗ ಬರುತ್ತದೆ?"),
    ("bro file upload aagilla, size problem ideya?",        "kannada", "ಕಡತ ಅಪಲೋಡ್ ಆಗಿಲ್ಲ, ಗಾತ್ರದ ಸಮಸ್ಯೆ ಇದೆಯಾ?"),
    ("discount code work aagilla yaar",                     "kannada", "ರಿಯಾಯಿತಿ ಸಂಕೇತ ಕಾರ್ಯನಿರ್ವಹಿಸುತ್ತಿಲ್ಲ."),
    ("bro naale holiday ideya office ge?",                  "kannada", "ನಾಳೆ ಕಚೇರಿಗೆ ರಜೆ ಇದೆಯಾ?"),
    ("ivathu wfh confirm aagide",                           "kannada", "ಇಂದು ಮನೆಯಿಂದ ಕೆಲಸ ದೃಢಪಡಿಸಲಾಗಿದೆ."),
    ("bro system hang aagide restart madtheena",            "kannada", "ವ್ಯವಸ್ಥೆ ಸ್ಥಗಿತಗೊಂಡಿದೆ, ಮರುಪ್ರಾರಂಭಿಸಬೇಕೇ?"),
    ("payment screenshot kalsthini, check madanna",        "kannada", "ಭುಗತಾನದ ಚಿತ್ರವನ್ನು ಕಳಿಸುತ್ತೇನೆ, ಪರಿಶೀಲಿಸಿ."),
    ("bro call drop aagtha ide, bad network",               "kannada", "ಸಂಪರ್ಕ ಕಡಿತವಾಗುತ್ತಿದೆ, ಜಾಲ ಸಂಪರ್ಕ ಸರಿಯಿಲ್ಲ."),
    ("interview kal ide, wish madanna",                     "kannada", "ನಾಳೆ ಸಂದರ್ಶನ ಇದೆ, ಶುಭ ಹಾರೈಸಿ."),
]

MY_TANGLISH = [
    ("bro naalaiku office varuveengala?",                   "tamil", "நாளைக்கு அலுவலகத்திற்கு வருவீர்களா?"),
    ("app update pannanum, reminder kudukuren",             "tamil", "பயன்பாட்டை புதுப்பிக்க வேண்டும், நினைவூட்டல் தருகிறேன்."),
    ("bro account block aagiduchi, enna pannrathu?",        "tamil", "கணக்கு தடுக்கப்பட்டுவிட்டது, என்ன செய்வது?"),
    ("inaiku deadline da, konjam help pannunga",            "tamil", "இன்று இறுதி தேதி, கொஞ்சம் உதவுங்கள்."),
    ("net illama meeting join pannanum",                    "tamil", "இணையம் இல்லாமல் கூட்டத்தில் சேர வேண்டும்."),
    ("bro refund eppo varum?",                              "tamil", "பணத்தை திரும்ப எப்போது கிடைக்கும்?"),
    ("file upload aagala, size problem irukka?",            "tamil", "கோப்பு பதிவேற்றம் ஆகவில்லை, அளவு பிரச்சனை இருக்கிறதா?"),
    ("discount code work aagala yaar",                      "tamil", "தள்ளுபடி குறியீடு வேலை செய்யவில்லை."),
    ("bro naalaiku holiday irukka office ku?",              "tamil", "நாளைக்கு அலுவலகத்திற்கு விடுமுறை இருக்கிறதா?"),
    ("inaiku wfh confirm aagiduchi",                        "tamil", "இன்று வீட்டிலிருந்து பணி உறுதிப்படுத்தப்பட்டுவிட்டது."),
    ("bro system hang aagiduchi restart pannalama?",        "tamil", "கணினி செயலிழந்துவிட்டது, மறுதொடக்கம் செய்யலாமா?"),
    ("payment screenshot anupuren, check pannunga",         "tamil", "பணம் செலுத்திய தெளிவுப்படம் அனுப்புகிறேன், சரிபாருங்கள்."),
    ("bro call drop aaguthu, network seri illa",            "tamil", "தொடர்பு துண்டிக்கப்படுகிறது, பிணையம் சரியில்லை."),
    ("interview naalaiku iruku, wish pannunga",             "tamil", "நாளைக்கு நேர்காணல் இருக்கிறது, வாழ்த்துங்கள்."),
    ("bro inaiku presentation iruku, nervous aa iruku",     "tamil", "இன்று விளக்கக்காட்சி இருக்கிறது, பதட்டமாக இருக்கிறது."),
]

MY_INFORMAL_ENGLISH = [
    ("bruh the app keeps crashing every time i open it",    "hindi",   "अनुप्रयोग हर बार खोलने पर बंद हो जाता है।"),
    ("ngl this update made everything worse",               "kannada", "솔직히 말하면 ಈ ನವೀಕರಣ ಎಲ್ಲವನ್ನೂ ಇನ್ನಷ್ಟು ಹದಗೆಡಿಸಿದೆ."),
    ("fr why is the server down again",                     "tamil",   "சேவையகம் ஏன் மீண்டும் செயலிழந்தது என்று புரியவில்லை."),
    ("lowkey stressed about the deadline tbh",              "telugu",  "గడువు గురించి నిజంగా ఒత్తిడిగా అనిపిస్తోంది."),
    ("cant believe the payment failed again smh",           "hindi",   "विश्वास नहीं होता कि भुगतान फिर से विफल हो गया।"),
    ("anyone else feel like this app is a complete mess?",  "kannada", "ಈ ಅನ್ವಯ ಸಂಪೂರ್ಣ ಅಸ್ತವ್ಯಸ್ತವಾಗಿದೆ ಎಂದು ಬೇರೆ ಯಾರಿಗಾದರೂ ಅನಿಸುತ್ತಿದೆಯಾ?"),
]

MY_NATIVE_HINDI = [
    ("मेरा अकाउंट ब्लॉक हो गया है",                        "kannada", "ನನ್ನ ಖಾತೆ ನಿರ್ಬಂಧಿಸಲ್ಪಟ್ಟಿದೆ."),
    ("कृपया मीटिंग लिंक भेजें",                             "tamil",   "தயவுசெய்து சந்திப்பு இணைப்பை அனுப்புங்கள்."),
    ("फ़ाइल अपलोड नहीं हो रही है",                          "telugu",  "దస్త్రం అప్‌లోడ్ అవుటలేదు."),
    ("रिफंड कब तक मिलेगा?",                                 "malayalam","തിരിച்ചടവ് എപ്പോൾ ലഭിക്കും?"),
    ("पेमेंट फेल हो गई, दोबारा कोशिश करें",                 "bengali", "অর্থপ্রদান ব্যর্থ হয়েছে, আবার চেষ্টা করুন।"),
]

MY_NATIVE_KANNADA = [
    ("ನನ್ನ ಪಾಸ್‌ವರ್ಡ್ ಮರೆತು ಹೋಗಿದೆ",                       "hindi",   "मैं अपना पासवर्ड भूल गया हूं।"),
    ("ದಯವಿಟ್ಟು ಸಭೆಯ ಕೊಂಡಿ ಕಳಿಸಿ",                           "tamil",   "தயவுசெய்து சந்திப்பு இணைப்பை அனுப்புங்கள்."),
    ("ಅನ್ವಯ ತೆರೆದಾಗ ಕಾರ್ಯನಿಲ್ಲಿಸುತ್ತದೆ",                     "telugu",  "అనువర్తనం తెరిచినప్పుడు పని చేయడం ఆగిపోతోంది."),
    ("ಜಾಲ ಸಮಸ್ಯೆಯಿಂದ ಸಂಪರ್ಕ ಕಡಿತವಾಯಿತು",                    "hindi",   "नेटवर्क समस्या के कारण संपर्क टूट गया।"),
    ("ಹಣ ವಾಪಸಾತಿ ಇನ್ನೂ ಬಂದಿಲ್ಲ",                            "malayalam","തിരിച്ചടവ് ഇതുവരെ ലഭിച്ചിട്ടില്ല."),
]

MY_IDIOMS_AND_SLANG = [
    ("bite the bullet and just submit it",              "hindi",   "हिम्मत करके इसे जमा कर दो।"),
    ("he spilled the beans about the project",          "kannada", "ಅವನು ಯೋಜನೆಯ ರಹಸ್ಯವನ್ನು ಬಹಿರಂಗಪಡಿಸಿದನು."),
    ("we're back to square one",                        "tamil",   "நாம் மீண்டும் ஆரம்பத்திலிருந்து தொடங்க வேண்டும்."),
    ("stop beating around the bush",                    "telugu",  "విషయాన్ని తిరోగమించకుండా నేరుగా చెప్పండి."),
    ("let's kill two birds with one stone",             "hindi",   "एक काम से दो काम निपटा लेते हैं।"),
    ("he's under the weather today",                    "kannada", "ಅವನು ಇಂದು ಅನಾರೋಗ್ಯದಿಂದ ಬಳಲುತ್ತಿದ್ದಾನೆ."),
    ("this project is on the back burner",              "tamil",   "இந்த திட்டம் தற்காலிகமாக ஒத்திவைக்கப்பட்டுள்ளது."),
    ("pull my leg, that cant be true",                  "hindi",   "मुझे मूर्ख मत बनाओ, यह सच नहीं हो सकता।"),
    ("the ball is in your court now",                   "telugu",  "ఇప్పుడు నిర్ణయం మీ చేతిలో ఉంది."),
    ("we need to get the ball rolling",                 "kannada", "ನಾವು ಕಾರ್ಯವನ್ನು ಪ್ರಾರಂಭಿಸಬೇಕು."),
    ("that ship has sailed",                            "tamil",   "அந்த வாய்ப்பு இப்போது கடந்துவிட்டது."),
    ("it's not rocket science",                         "hindi",   "यह कोई बहुत कठिन काम नहीं है।"),
    ("this whole situation is giving me anxiety fr",    "kannada", "ಈ ಇಡೀ ಪರಿಸ್ಥಿತಿ ನನಗೆ ನಿಜವಾಗಿಯೂ ಆತಂಕ ಉಂಟುಮಾಡುತ್ತಿದೆ."),
    ("that meeting was an absolute vibe killer",        "tamil",   "அந்த கூட்டம் முழு மனநிலையையும் கெடுத்துவிட்டது."),
    ("im dead, this bug has been here since day one",   "hindi",   "मैं हैरान हूं, यह त्रुटि शुरू से ही मौजूद थी।"),
    ("the new ui hits different tbh",                   "telugu",  "నిజంగా చెప్పాలంటే కొత్త డిజైన్ చాలా భిన్నంగా అనిపిస్తోంది."),
    ("this deadline is not it chief",                   "hindi",   "यह समय सीमा बिल्कुल उचित नहीं है।"),
    ("we been knew this would happen",                  "kannada", "ಇದು ಆಗುತ್ತದೆ ಎಂದು ನಮಗೆ ಮೊದಲೇ ತಿಳಿದಿತ್ತು."),
    ("the app is straight up trash rn",                 "tamil",   "இப்போது இந்த பயன்பாடு முற்றிலும் பயனற்றதாக உள்ளது."),
    ("iykyk this server is always down mondays",        "hindi",   "जो जानते हैं वो जानते हैं, यह सर्वर सोमवार को हमेशा बंद रहता है।"),
]

MY_NATIVE_TAMIL = [
    ("என் கடவுச்சொல் மறந்துவிட்டது",                       "hindi",   "मैं अपना पासवर्ड भूल गया हूं।"),
    ("தயவுசெய்து சந்திப்பு இணைப்பை அனுப்புங்கள்",            "kannada", "ದಯವಿಟ್ಟು ಸಭೆಯ ಕೊಂಡಿ ಕಳಿಸಿ."),
    ("பிணையப் பிரச்சனையால் தொடர்பு துண்டிக்கப்பட்டது",       "telugu",  "నెట్‌వర్క్ సమస్య వల్ల సంబంధం తెగిపోయింది."),
    ("பணத்தை திரும்ப இன்னும் கிடைக்கவில்லை",                "hindi",   "धनवापसी अभी तक नहीं मिली है।"),
    ("பயன்பாடு திறக்கும்போது செயலிழக்கிறது",                 "bengali", "অ্যাপ্লিকেশন খোলার সময় বন্ধ হয়ে যাচ্ছে।"),
]

MY_NATIVE_TELUGU = [
    ("నా పాస్‌వర్డ్ మర్చిపోయాను",                           "hindi",   "मैं अपना पासवर्ड भूल गया हूं।"),
    ("దయచేసి సభా లింక్ పంపండి",                             "tamil",   "தயவுசெய்து சந்திப்பு இணைப்பை அனுப்புங்கள்."),
    ("చెల్లింపు విఫలమైంది, మళ్ళీ ప్రయత్నించండి",              "kannada", "ಪಾವತಿ ವಿಫಲವಾಗಿದೆ, ಮತ್ತೆ ಪ್ರಯತ್ನಿಸಿ."),
    ("నెట్‌వర్క్ సమస్య వల్ల సంబంధం తెగిపోయింది",              "malayalam","നെറ്റ്‌വർക്ക് പ്രശ്‌നം കാരണം ബന്ധം വിച്ഛേദിക്കപ്പെട്ടു."),
    ("తిరిగి చెల్లింపు ఇంకా రాలేదు",                         "hindi",   "धनवापसी अभी तक नहीं मिली है।"),
]

MY_NATIVE_MALAYALAM = [
    ("എന്റെ പാസ്‌വേഡ് മറന്നുപോയി",                          "hindi",   "मैं अपना पासवर्ड भूल गया हूं।"),
    ("ദയവായി സഭാ ലിങ്ക് അയയ്ക്കുക",                          "tamil",   "தயவுசெய்து சந்திப்பு இணைப்பை அனுப்புங்கள்."),
    ("പണമടയ്ക്കൽ പരാജയപ്പെട്ടു, വീണ്ടും ശ്രമിക്കുക",          "kannada", "ಪಾವತಿ ವಿಫಲವಾಗಿದೆ, ಮತ್ತೆ ಪ್ರಯತ್ನಿಸಿ."),
    ("നെറ്റ്‌വർക്ക് പ്രശ്‌നം കാരണം ബന്ധം വിച്ഛേദിക്കപ്പെട്ടു", "telugu",  "నెట్‌వర్క్ సమస్య వల్ల సంబంధం తెగిపోయింది."),
    ("തിരിച്ചടവ് ഇതുവരെ ലഭിച്ചിട്ടില്ല",                     "hindi",   "धनवापसी अभी तक नहीं मिली है।"),
]

MY_NATIVE_BENGALI = [
    ("আমার পাসওয়ার্ড ভুলে গেছি",                            "hindi",   "मैं अपना पासवर्ड भूल गया हूं।"),
    ("অনুগ্রহ করে সভার লিঙ্ক পাঠান",                          "tamil",   "தயவுசெய்து சந்திப்பு இணைப்பை அனுப்புங்கள்."),
    ("অর্থপ্রদান ব্যর্থ হয়েছে, আবার চেষ্টা করুন",             "kannada", "ಪಾವತಿ ವಿಫಲವಾಗಿದೆ, ಮತ್ತೆ ಪ್ರಯತ್ನಿಸಿ."),
    ("নেটওয়ার্ক সমস্যার কারণে সংযোগ বিচ্ছিন্ন হয়েছে",        "telugu",  "నెట్‌వర్క్ సమస్య వల్ల సంబంధం తెగిపోయింది."),
    ("ফেরত অর্থ এখনও আসেনি",                                 "hindi",   "धनवापसी अभी तक नहीं मिली है।"),
]

print("✅ All new sentences loaded!")
print(f"  Hinglish:          {len(MY_HINGLISH)}")
print(f"  Kanglish:          {len(MY_KANGLISH)}")
print(f"  Tanglish:          {len(MY_TANGLISH)}")
print(f"  Informal English:  {len(MY_INFORMAL_ENGLISH)}")
print(f"  Native Hindi:      {len(MY_NATIVE_HINDI)}")
print(f"  Native Kannada:    {len(MY_NATIVE_KANNADA)}")
print(f"  Native Tamil:      {len(MY_NATIVE_TAMIL)}")
print(f"  Native Telugu:     {len(MY_NATIVE_TELUGU)}")
print(f"  Native Malayalam:  {len(MY_NATIVE_MALAYALAM)}")
print(f"  Native Bengali:    {len(MY_NATIVE_BENGALI)}")
total = (len(MY_HINGLISH) + len(MY_KANGLISH) + len(MY_TANGLISH) +
         len(MY_INFORMAL_ENGLISH) + len(MY_NATIVE_HINDI) + len(MY_NATIVE_KANNADA) +
         len(MY_NATIVE_TAMIL) + len(MY_NATIVE_TELUGU) + len(MY_NATIVE_MALAYALAM) +
         len(MY_NATIVE_BENGALI))
print(f"  ─────────────────────────")
print(f"  Subtotal new:      {total}")

✅ All new sentences loaded!
  Hinglish:          22
  Kanglish:          16
  Tanglish:          15
  Informal English:  6
  Native Hindi:      5
  Native Kannada:    5
  Native Tamil:      5
  Native Telugu:     5
  Native Malayalam:  5
  Native Bengali:    5
  ─────────────────────────
  Subtotal new:      89


In [ ]:
EVAL_DATASET_EXTENDED = []
EVAL_DATASET_EXTENDED.extend(EVAL_DATASET)
EVAL_DATASET_EXTENDED.extend(flores_eval_entries)
EVAL_DATASET_EXTENDED.extend(MY_HINGLISH)
EVAL_DATASET_EXTENDED.extend(MY_KANGLISH)
EVAL_DATASET_EXTENDED.extend(MY_TANGLISH)
EVAL_DATASET_EXTENDED.extend(MY_INFORMAL_ENGLISH)
EVAL_DATASET_EXTENDED.extend(MY_NATIVE_HINDI)
EVAL_DATASET_EXTENDED.extend(MY_NATIVE_KANNADA)
EVAL_DATASET_EXTENDED.extend(MY_NATIVE_TAMIL)
EVAL_DATASET_EXTENDED.extend(MY_NATIVE_TELUGU)
EVAL_DATASET_EXTENDED.extend(MY_NATIVE_MALAYALAM)
EVAL_DATASET_EXTENDED.extend(MY_NATIVE_BENGALI)
EVAL_DATASET_EXTENDED.extend(MY_IDIOMS_AND_SLANG)

print(f"Original dataset:      {len(EVAL_DATASET)}")
print(f"FLORES added:          {len(flores_eval_entries)}")
print(f"New sentences added:   {total}")
print(f"──────────────────────────────")
print(f"Total:                 {len(EVAL_DATASET_EXTENDED)}")

Original dataset:      60
FLORES added:          30
New sentences added:   89
──────────────────────────────
Total:                 199


In [ ]:
!pip install -q deep-translator

from deep_translator import GoogleTranslator
import time

GT_LANG_MAP = {
    "hindi":     "hi",
    "kannada":   "kn",
    "tamil":     "ta",
    "telugu":    "te",
    "bengali":   "bn",
    "marathi":   "mr",
    "malayalam": "ml",
    "gujarati":  "gu",
    "punjabi":   "pa",
    "odia":      "or",
}

def google_translate(text: str, target_language: str) -> str:
    try:
        tgt = GT_LANG_MAP.get(target_language, "hi")
        result = GoogleTranslator(source='auto', target=tgt).translate(text)
        time.sleep(0.3)
        return result
    except Exception as e:
        return f"[ERROR: {e}]"

# Quick test
print(google_translate("bhai server down hai kya?", "hindi"))
print(google_translate("idk man it's raining cats and dogs", "hindi"))
print("✅ Google Translate ready")

bhai server down hai kya?
मुझे नहीं पता यार, कुत्तों और बिल्लियों की बारिश हो रही है
✅ Google Translate ready


In [ ]:
# ── Run Evaluation: v2 vs Google Translate vs IndicTrans2 Raw ──────────────
import time

eval_results = []
ctx = ConversationContext(max_history=0)  # no context — fair comparison

print(f"Running evaluation on {len(EVAL_DATASET_EXTENDED)} sentences...")
print("This will take a few minutes.\n")

for i, (text, target_lang, reference) in enumerate(EVAL_DATASET_EXTENDED):
    print(f"[{i+1}/{len(EVAL_DATASET_EXTENDED)}] {text[:50]}...")
    tgt_code = LANG_CODE.get(target_lang)

    # ── System 1: Your v2 pipeline ──
    try:
        v2_out = agentic_translate_v2(
            user_text=text,
            target_language=target_lang,
            context=ctx,
            speaker="User",
            eval_mode=False
        )
        v2_translation = v2_out["translation"]
        v2_normalized  = v2_out["normalized_english"]
        norm_skipped   = v2_out["normalization_skipped"]
    except Exception as e:
        v2_translation = f"[ERROR: {e}]"
        v2_normalized  = text
        norm_skipped   = True

    # ── System 2: IndicTrans2 raw (no normalization) ──
    try:
        raw_translation = translate_with_indictrans([text], tgt_lang=tgt_code)[0]
    except Exception as e:
        raw_translation = f"[ERROR: {e}]"

    # ── System 3: Google Translate ──
    gt_translation = google_translate(text, target_lang)

    # ── BERTScore — all three vs human reference ──
    bs_v2  = compute_bertscore([v2_translation],  [reference], lang="en")
    bs_raw = compute_bertscore([raw_translation], [reference], lang="en")
    bs_gt  = compute_bertscore([gt_translation],  [reference], lang="en")

    eval_results.append({
        "input":           text,
        "target_language": target_lang,
        "reference":       reference,
        "v2_normalized":   v2_normalized,
        "norm_skipped":    norm_skipped,
        "v2_translation":  v2_translation,
        "raw_translation": raw_translation,
        "gt_translation":  gt_translation,
        "bertscore_v2":    round(bs_v2["f1"][0],  4),
        "bertscore_raw":   round(bs_raw["f1"][0], 4),
        "bertscore_gt":    round(bs_gt["f1"][0],  4),
    })

    time.sleep(0.5)

print(f"\n✅ Evaluation complete — {len(eval_results)} results stored in eval_results")

Running evaluation on 199 sentences...
This will take a few minutes.

[1/60] bhai server down hai kya?...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[2/60] yaar server firse down ho gaya...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[3/60] pls thoda jaldi karo, urgent hai...


...

[198/60] the app is straight up trash rn...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[199/60] iykyk this server is always down mondays...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



✅ Evaluation complete — 199 results stored in eval_results


In [ ]:
# ── Evaluation Results Summary ─────────────────────────────────────────────
import statistics

categories = {
    "Hinglish romanized":    [r for r in eval_results if r["target_language"] == "hindi"    and any(w in r["input"] for w in ["bhai","yaar","kya","hai","raha","nahi","aaj"])],
    "Kanglish romanized":    [r for r in eval_results if r["target_language"] == "kannada"  and any(w in r["input"] for w in ["maadi","illa","hogbeku","swalpa","aitu","aagide"])],
    "Tanglish romanized":    [r for r in eval_results if r["target_language"] == "tamil"    and any(w in r["input"] for w in ["irukku","pannunga","varala","aaguthu","konjam"])],
    "Native script":         [r for r in eval_results if not any(c.isascii() for c in r["input"] if c.isalpha())],
    "Informal English":      [r for r in eval_results if r["target_language"] == "hindi"    and all(c.isascii() for c in r["input"])],
    "Idioms & slang": [r for r in eval_results if any(w in r["input"].lower() for w in [
    "break a leg","raining cats","arm and a leg","piece of cake","last straw",
    "blue moon","hit the sack","ngl","fr ","lowkey","highkey","no cap","bruh",
    "kinda mid","slaps","wild af","bite the bullet","spilled the beans",
    "square one","beating around","two birds","under the weather","back burner",
    "pull my leg","ball is in","get the ball","that ship","rocket science",
    "vibe killer","hits different","not it chief","been knew","straight up trash","iykyk"
])],
}

print("=" * 80)
print("📊 EVALUATION RESULTS — BERTScore F1 (higher = better)")
print("=" * 80)
print(f"{'Category':<25} {'Count':>5} {'v2 Pipeline':>12} {'IndicTrans2 Raw':>15} {'Google Translate':>17}")
print("-" * 80)

all_v2, all_raw, all_gt = [], [], []

for cat_name, cat_results in categories.items():
    if not cat_results:
        continue
    v2_scores  = [r["bertscore_v2"]  for r in cat_results]
    raw_scores = [r["bertscore_raw"] for r in cat_results]
    gt_scores  = [r["bertscore_gt"]  for r in cat_results]

    mean_v2  = statistics.mean(v2_scores)
    mean_raw = statistics.mean(raw_scores)
    mean_gt  = statistics.mean(gt_scores)

    all_v2  += v2_scores
    all_raw += raw_scores
    all_gt  += gt_scores

    winner = "✅" if mean_v2 >= max(mean_raw, mean_gt) else "⚠️"
    print(f"{winner} {cat_name:<23} {len(cat_results):>5} {mean_v2:>12.4f} {mean_raw:>15.4f} {mean_gt:>17.4f}")

print("-" * 80)
print(f"{'OVERALL':<25} {len(eval_results):>5} {statistics.mean(all_v2):>12.4f} {statistics.mean(all_raw):>15.4f} {statistics.mean(all_gt):>17.4f}")
print("=" * 80)

# Improvement summary
v2_vs_raw = statistics.mean(all_v2) - statistics.mean(all_raw)
v2_vs_gt  = statistics.mean(all_v2) - statistics.mean(all_gt)
print(f"\n  v2 pipeline vs IndicTrans2 raw:   {'+' if v2_vs_raw >= 0 else ''}{v2_vs_raw:.4f}")
print(f"  v2 pipeline vs Google Translate:  {'+' if v2_vs_gt  >= 0 else ''}{v2_vs_gt:.4f}")

📊 EVALUATION RESULTS — BERTScore F1 (higher = better)
Category                  Count  v2 Pipeline IndicTrans2 Raw  Google Translate
--------------------------------------------------------------------------------
✅ Hinglish romanized         29       0.9558          0.9427            0.6877
✅ Kanglish romanized         12       0.9549          0.9548            0.7603
✅ Tanglish romanized          9       0.9603          0.9586            0.7837
⚠️ Native script              89       0.9499          0.8784            0.9714
✅ Informal English           45       0.9459          0.9295            0.7681
✅ Idioms & slang             25       0.9499          0.9404            0.9415
--------------------------------------------------------------------------------
OVERALL                     199       0.9506          0.9136            0.8645

  v2 pipeline vs IndicTrans2 raw:   +0.0370
  v2 pipeline vs Google Translate:  +0.0861


In [ ]:
examples = [
    # (label, input, target_language)

    # --- Idioms (English → various) ---
    ("Idiom", "idk man, it's raining cats and dogs",       "hindi"),
    ("Idiom", "break a leg for your interview",             "kannada"),
    ("Idiom", "costs an arm and a leg",                     "tamil"),
    ("Idiom", "hit the sack, I'm tired",                   "telugu"),
    ("Idiom", "once in a blue moon",                        "bengali"),
    ("Idiom", "this is the last straw",                     "marathi"),

    # --- Slang (English → various) ---
    ("Slang", "bruh this is wild af",                       "tamil"),
    ("Slang", "ngl that was kinda mid",                     "kannada"),
    ("Slang", "fr this app slaps",                          "bengali"),
    ("Slang", "lowkey I'm done with this",                  "telugu"),
    ("Slang", "highkey this is annoying",                   "marathi"),
    ("Slang", "no cap, that's impressive",                  "hindi"),

    # --- Hinglish (Hindi) → other languages ---
    ("Hinglish", "bhai server down hai kya?",               "tamil"),
    ("Hinglish", "OTP nahi aa raha, kya karu?",            "kannada"),
    ("Hinglish", "yaar server firse down ho gaya",          "telugu"),
    ("Hinglish", "pls thoda jaldi karo, urgent hai",        "bengali"),

    # --- Kanglish (Kannada) → other languages ---
    ("Kanglish", "bro swalpa adjust maadi, urgent ide",     "hindi"),
    ("Kanglish", "payment aagide but receipt baralla",      "tamil"),
    ("Kanglish", "ninna night app hang aitu",               "telugu"),
    ("Kanglish", "kelsa complete madidini, pls review",     "bengali"),

    # --- Tanglish (Tamil) → other languages ---
    ("Tanglish", "bro konjam wait pannunga, net slow",      "hindi"),
    ("Tanglish", "app open panna crash aaguthu",            "kannada"),
    ("Tanglish", "enaku OTP varala, help pannunga",         "telugu"),
    ("Tanglish", "server down ah? romba worst",             "bengali"),

    # --- Broken English → various ---
    ("Broken English", "cant login, pw reset not working",  "tamil"),
    ("Broken English", "app crash when open only",          "kannada"),
    ("Broken English", "net slow so msg late",              "hindi"),
    ("Broken English", "i no understand why error coming",  "bengali"),
    ("Broken English", "payment done but not reflecting",   "telugu"),
    ("Broken English", "why my account lock??",             "marathi"),

    # --- Native Script → different language ---
    ("Native Script", "सर्वर डाउन हो गया है",              "tamil"),    # Hindi → Tamil
    ("Native Script", "मेरा भुगतान अटक गया है",             "kannada"),  # Hindi → Kannada
    ("Native Script", "ಸರ್ವರ್ ಡೌನ್ ಆಗಿದೆ",                  "hindi"),    # Kannada → Hindi
    ("Native Script", "ದಯವಿಟ್ಟು ಸಹಾಯ ಮಾಡಿ",                 "telugu"),   # Kannada → Telugu
    ("Native Script", "சர்வர் டவுன் ஆகிவிட்டது",             "hindi"),    # Tamil → Hindi
    ("Native Script", "பணம் திரும்பவில்லை",                   "kannada"),  # Tamil → Kannada
    ("Native Script", "సర్వర్ డౌన్ అయింది",                   "tamil"),    # Telugu → Tamil
    ("Native Script", "చెల్లింపు నిలిచిపోయింది",               "bengali"),  # Telugu → Bengali
    ("Native Script", "সার্ভার ডাউন হয়েছে",                   "hindi"),    # Bengali → Hindi
    ("Native Script", "পেমেন্ট আটকে গেছে",                    "tamil"),    # Bengali → Tamil
    ("Native Script", "سর্ভার ডাউন হয়েছে",                   "telugu"),   # Bengali → Telugu
    ("Native Script", "سر्व्हर डाउन झाला आहे",               "kannada"),  # Marathi → Kannada
    ("Native Script", "സർവർ ഡൗൺ ആയി",                       "hindi"),    # Malayalam → Hindi
    ("Native Script", "ਸਰਵਰ ਡਾਊਨ ਹੋ ਗਿਆ",                    "tamil"),    # Punjabi → Tamil
]

ctx_demo = ConversationContext(max_history=0)

print("=" * 115)
print("📋 QUALITATIVE COMPARISON — v2 Pipeline vs IndicTrans2 Raw vs Google Translate")
print("=" * 115)

current_category = ""
for label, text, lang in examples:
    if label != current_category:
        current_category = label
        print(f"\n{'─' * 115}")
        print(f"  📌 {label.upper()}")
        print(f"{'─' * 115}")

    # v2 pipeline
    v2_out = agentic_translate_v2(
        user_text=text,
        target_language=lang,
        context=ctx_demo,
        speaker="User"
    )
    v2_norm  = v2_out["normalized_english"]
    v2_trans = v2_out["translation"]
    skipped  = v2_out["normalization_skipped"]

    # IndicTrans2 raw
    tgt_code  = LANG_CODE[lang]
    raw_trans = translate_with_indictrans([text], tgt_lang=tgt_code)[0]

    # Google Translate
    gt_trans = google_translate(text, lang)

    # Detect source language for label
    src_lang = v2_out["detected_language"].capitalize()

    print(f"\n  INPUT [{src_lang} → {lang.capitalize()}]: {text}")
    if not skipped:
        print(f"  Normalized:          {v2_norm}")
    print(f"  ✅ v2 Pipeline:      {v2_trans}")
    print(f"  ⚙️  IndicTrans2 Raw:  {raw_trans}")
    print(f"  🌐 Google Translate: {gt_trans}")

print(f"\n{'=' * 115}")

📋 QUALITATIVE COMPARISON — v2 Pipeline vs IndicTrans2 Raw vs Google Translate

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  📌 IDIOM
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

  INPUT [English → Hindi]: idk man, it's raining cats and dogs
  Normalized:          It's raining very heavily.
  ✅ v2 Pipeline:      बहुत बारिश हो रही है।
  ⚙️  IndicTrans2 Raw:  मूर्ख आदमी, बिल्लियों और कुत्तों की बारिश हो रही है
  🌐 Google Translate: मुझे नहीं पता यार, कुत्तों और बिल्लियों की बारिश हो रही है

  INPUT [English → Kannada]: break a leg for your interview
  Normalized:          Good luck with your interview.
  ✅ v2 Pipeline:      ನಿಮ್ಮ ಸಂದರ್ಶನಕ್ಕೆ ಶುಭವಾಗಲಿ.
  ⚙️  IndicTrans2 Raw:  ನಿಮ್ಮ ಸಂದರ್ಶನಕ್ಕಾಗಿ ಒಂದು ಕಾಲು ಮುರಿಯಿರಿ
  🌐 Google Translate: ನಿಮ್ಮ ಸಂದರ್ಶನಕ್ಕಾಗಿ ಕಾಲು ಮುರಿಯಿರಿ

  INPUT [English → Tamil]: costs an arm and a leg
  Normalized:          This c

In [ ]:
# Paper stats summary
successful = [r for r in eval_results if "ERROR" not in r["v2_translation"]]
norm_triggered = [r for r in eval_results if not r["norm_skipped"]]

print("=== PAPER STATS ===")
print(f"Total evaluation sentences:     {len(eval_results)}")
print(f"Languages covered:              {len(set(r['target_language'] for r in eval_results))}")
print(f"Normalization triggered:        {len(norm_triggered)}/{len(eval_results)} ({100*len(norm_triggered)/len(eval_results):.1f}%)")
print(f"Normalization skipped:          {len(eval_results)-len(norm_triggered)}/{len(eval_results)}")
print(f"\nBERTScore F1 — v2 Pipeline:     {statistics.mean([r['bertscore_v2'] for r in eval_results]):.4f}")
print(f"BERTScore F1 — IndicTrans2 Raw: {statistics.mean([r['bertscore_raw'] for r in eval_results]):.4f}")
print(f"BERTScore F1 — Google Translate:{statistics.mean([r['bertscore_gt'] for r in eval_results]):.4f}")
print(f"\nImprovement over IndicTrans2:   +{statistics.mean([r['bertscore_v2']-r['bertscore_raw'] for r in eval_results]):.4f}")
print(f"Improvement over Google:        +{statistics.mean([r['bertscore_v2']-r['bertscore_gt'] for r in eval_results]):.4f}")
print(f"\nDetection methods used:")
for method in ['script', 'indiclid', 'fallback', 'wordlist']:
    count = sum(1 for r in eval_results if r.get('detection_method') == method)
    if count: print(f"  {method}: {count}")

=== PAPER STATS ===
Total evaluation sentences:     179
Languages covered:              10
Normalization triggered:        167/179 (93.3%)
Normalization skipped:          12/179

BERTScore F1 — v2 Pipeline:     0.9481
BERTScore F1 — IndicTrans2 Raw: 0.9168
BERTScore F1 — Google Translate:0.8805

Improvement over IndicTrans2:   +0.0313
Improvement over Google:        +0.0676

Detection methods used:


In [ ]:
!pip install -q fastapi uvicorn nest-asyncio pyngrok

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("3ADEYqubgvOYue3afrPS4oelzzz_7J36BdaCcR7vk2QDi6Mt8")

In [ ]:
!pip install -q fastapi uvicorn nest-asyncio pyngrok

import nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok
import uvicorn
import threading

nest_asyncio.apply()

app = FastAPI()

class TranslateRequest(BaseModel):
    text: str
    target_language: str

@app.post("/translate")
def translate(req: TranslateRequest):
    try:
        ctx = ConversationContext(max_history=0)
        result = agentic_translate_v2(
            user_text=req.text,
            target_language=req.target_language,
            context=ctx,
            speaker="User"
        )
        return {"translation": result["translation"]}
    except Exception as e:
        return {"translation": req.text, "error": str(e)}

@app.get("/health")
def health():
    return {"status": "ok"}

# Start ngrok tunnel
public_url = ngrok.connect(8000)
url = public_url.public_url
print(f"\n✅ Public URL: {url}")
print(f"   Use this in application.properties: translator.api.url={url}\n")

# Start server
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run, daemon=True)
thread.start()


✅ Public URL: https://unparticipated-lecia-forcefully.ngrok-free.dev
   Use this in application.properties: translator.api.url=https://unparticipated-lecia-forcefully.ngrok-free.dev

